# 04 — Feature-matrix construction

Public source-only reproducibility notebook. Outputs and execution history were removed. No credentials, patient-level data, row-level predictions, or row-level SHAP values are included. Execution requires credentialed access to the eICU Collaborative Research Database and an authorized Google Cloud project.


In [ ]:
import os
from pathlib import Path

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or input("Enter your Google Cloud project ID: ").strip()
WORK_DATASET_NAME = os.environ.get("AKI_DATASET_ID", "aki_jcmc_v2")
SOURCE_DATASET = os.environ.get("EICU_SOURCE_DATASET", "physionet-data.eicu_crd")
BQ_LOCATION = os.environ.get("BIGQUERY_LOCATION", "US")
OUTPUT_ROOT = os.environ.get("AKI_OUTPUT_ROOT", "/content/AKI_JCMC_V2_PUBLIC_RUN")
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
TARGET_DATASET = f"{PROJECT_ID}.{WORK_DATASET_NAME}"

print("Target dataset:", TARGET_DATASET)
print("Output root:", OUTPUT_ROOT)


# AKI V2 Feature Matrix Runner v1.0

Bu notebook, ilk 12 saatlik predictor matrisini BigQuery içinde oluşturur. Hasta düzeyindeki matris Drive'a kaydedilmez ve paylaşılmaz.

In [ ]:
from google.colab import auth, drive
from google.cloud import bigquery
import pandas as pd, os, json, hashlib, zipfile
from IPython.display import display

auth.authenticate_user()
drive.mount('/content/drive')
PROJECT_ID = globals().get("PROJECT_ID") or os.environ.get("GOOGLE_CLOUD_PROJECT") or input("Enter your Google Cloud project ID: ").strip()
SOURCE_DATASET = 'physionet-data.eicu_crd'
TARGET_DATASET = f"{PROJECT_ID}.{WORK_DATASET_NAME}"
BQ_LOCATION = 'US'
DRIVE_OUTPUT_DIR = f'{OUTPUT_ROOT}/04_FEATURE_MATRIX_OUTPUTS'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
print('Ready:', PROJECT_ID, TARGET_DATASET, BQ_LOCATION)


In [ ]:
def render_sql(sql):
    return (sql.replace('{{SOURCE_DATASET}}', SOURCE_DATASET)
               .replace('{{TARGET_DATASET}}', TARGET_DATASET))

def run_ddl(label, sql):
    print('Running:', label)
    client.query(render_sql(sql), location=BQ_LOCATION).result()
    print('Completed:', label)

def run_aggregate(label, sql, filename):
    print('Running:', label)
    df = client.query(render_sql(sql), location=BQ_LOCATION).to_dataframe()
    display(df.head(200))
    path = os.path.join(DRIVE_OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print('Saved:', path, 'rows=', len(df))
    return df


## 01 Create Static Features V1

In [ ]:
SQL_01 = "-- Secure one-row-per-patient static feature table.\nCREATE OR REPLACE TABLE `{{TARGET_DATASET}}.feature_static_v1` AS\nSELECT\n  c.patientUnitStayID,\n  c.hospitalID,\n  TO_HEX(SHA256(CONCAT('AKI_V2_STAY|', CAST(c.patientUnitStayID AS STRING)))) AS id_row,\n  TO_HEX(SHA256(CONCAT('AKI_V2_HOSPITAL|', CAST(c.hospitalID AS STRING)))) AS group_hospital,\n  c.outcome_creatinine_stage23 AS label_stage23,\n\n  CASE WHEN p.age = '> 89' THEN 90 ELSE SAFE_CAST(p.age AS INT64) END AS x_age_years,\n  CASE\n    WHEN LOWER(TRIM(p.gender)) = 'female' THEN 'female'\n    WHEN LOWER(TRIM(p.gender)) = 'male' THEN 'male'\n    ELSE 'other_or_missing'\n  END AS x_sex,\n  CASE\n    WHEN p.admissionHeight BETWEEN 100 AND 250\n     AND p.admissionWeight BETWEEN 25 AND 300\n     AND SAFE_DIVIDE(p.admissionWeight, POW(p.admissionHeight / 100.0, 2)) BETWEEN 10 AND 80\n    THEN SAFE_DIVIDE(p.admissionWeight, POW(p.admissionHeight / 100.0, 2))\n  END AS x_bmi,\n  IF(p.admissionWeight BETWEEN 25 AND 300, p.admissionWeight, NULL) AS x_admission_weight_kg,\n  NULLIF(TRIM(p.unitType), '') AS x_unit_type,\n  NULLIF(TRIM(p.unitAdmitSource), '') AS x_unit_admit_source,\n  NULLIF(TRIM(p.hospitalAdmitSource), '') AS x_hospital_admit_source,\n  c.reference_creatinine AS x_reference_creatinine,\n  c.stage1_at_prediction AS x_stage1_at_landmark,\n\n  NULLIF(TRIM(p.ethnicity), '') AS audit_ethnicity,\n  NULLIF(TRIM(h.region), '') AS audit_hospital_region,\n  NULLIF(TRIM(h.numBedsCategory), '') AS audit_hospital_bed_category,\n  h.teachingStatus AS audit_hospital_teaching_status,\n  c.reference_offset AS audit_reference_offset\nFROM `{{TARGET_DATASET}}.cohort_outcome_v1` c\nJOIN `{{SOURCE_DATASET}}.patient` p\n  ON p.patientUnitStayID = c.patientUnitStayID\nLEFT JOIN `{{SOURCE_DATASET}}.hospital` h\n  ON h.hospitalID = c.hospitalID\nWHERE c.eligible_main_cohort = 1\n  AND c.deterministic_eligible_patient_stay_rank = 1;\n"
run_ddl('01_create_static_features_v1.sql', SQL_01)

In [ ]:
SQL_01_AUDIT = f"""
SELECT
  COUNT(*) AS table_rows,
  COUNT(DISTINCT patientUnitStayID) AS distinct_unit_stays,
  COUNT(DISTINCT id_row) AS distinct_row_keys,
  COUNT(DISTINCT hospitalID) AS hospitals,
  COUNT(DISTINCT group_hospital) AS hospital_group_keys,

  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  SAFE_DIVIDE(
    COUNTIF(label_stage23 = 1),
    COUNT(*)
  ) AS event_rate,

  COUNT(*) - COUNT(DISTINCT patientUnitStayID)
    AS duplicate_stay_rows,

  COUNTIF(label_stage23 IS NULL)
    AS missing_labels,

  COUNTIF(x_age_years IS NULL)
    AS missing_age,

  COUNTIF(x_sex IS NULL)
    AS missing_sex,

  COUNTIF(x_bmi IS NULL)
    AS missing_or_implausible_bmi,

  COUNTIF(x_admission_weight_kg IS NULL)
    AS missing_or_implausible_weight,

  COUNTIF(x_reference_creatinine IS NULL)
    AS missing_reference_creatinine,

  COUNTIF(x_stage1_at_landmark = 1)
    AS stage1_at_landmark,

  COUNTIF(audit_reference_offset > 360)
    AS reference_after_6h,

  MIN(audit_reference_offset)
    AS minimum_reference_offset,

  MAX(audit_reference_offset)
    AS maximum_reference_offset

FROM `{TARGET_DATASET}.feature_static_v1`;
"""

result_01_audit = run_aggregate(
    "01A_static_feature_integrity_audit.sql",
    SQL_01_AUDIT,
    "01A_static_feature_integrity_audit.csv"
)

## 02 Create Lab Features V1

In [ ]:
SQL_02 = "-- First-12-hour laboratory summaries after broad plausibility QC.\nCREATE OR REPLACE TABLE `{{TARGET_DATASET}}.feature_labs_v1` AS\nWITH main AS (\n  SELECT patientUnitStayID\n  FROM `{{TARGET_DATASET}}.cohort_outcome_v1`\n  WHERE eligible_main_cohort = 1\n    AND deterministic_eligible_patient_stay_rank = 1\n),\nraw AS (\n  SELECT\n    l.patientUnitStayID,\n    l.labID,\n    l.labResultOffset AS offset_min,\n    LOWER(TRIM(l.labName)) AS lab_name,\n    SAFE_CAST(l.labResult AS FLOAT64) AS raw_value\n  FROM main m\n  JOIN `{{SOURCE_DATASET}}.lab` l USING(patientUnitStayID)\n  WHERE l.labResultOffset BETWEEN 0 AND 720\n    AND l.labResult IS NOT NULL\n),\nmapped AS (\n  SELECT *,\n    CASE lab_name\n      WHEN 'creatinine' THEN 'creatinine'\n      WHEN 'bun' THEN 'bun'\n      WHEN 'bicarbonate' THEN 'bicarbonate'\n      WHEN 'potassium' THEN 'potassium'\n      WHEN 'sodium' THEN 'sodium'\n      WHEN 'chloride' THEN 'chloride'\n      WHEN 'glucose' THEN 'glucose_serum'\n      WHEN 'bedside glucose' THEN 'glucose_bedside'\n      WHEN 'calcium' THEN 'calcium'\n      WHEN 'magnesium' THEN 'magnesium'\n      WHEN 'phosphate' THEN 'phosphate'\n      WHEN 'hgb' THEN 'hgb'\n      WHEN 'hct' THEN 'hct'\n      WHEN 'platelets x 1000' THEN 'platelets'\n      WHEN 'wbc x 1000' THEN 'wbc'\n      WHEN 'albumin' THEN 'albumin'\n      WHEN 'total bilirubin' THEN 'bilirubin_total'\n      WHEN 'ast (sgot)' THEN 'ast'\n      WHEN 'alt (sgpt)' THEN 'alt'\n      WHEN 'pt - inr' THEN 'inr'\n      WHEN 'lactate' THEN 'lactate'\n      WHEN 'anion gap' THEN 'anion_gap'\n      ELSE NULL\n    END AS feature\n  FROM raw\n),\nclean AS (\n  SELECT patientUnitStayID, labID, offset_min, feature,\n    CASE feature\n      WHEN 'creatinine' THEN IF(raw_value BETWEEN 0.05 AND 40.0, raw_value, NULL)\n      WHEN 'bun' THEN IF(raw_value BETWEEN 1.0 AND 350.0, raw_value, NULL)\n      WHEN 'bicarbonate' THEN IF(raw_value BETWEEN 2.0 AND 80.0, raw_value, NULL)\n      WHEN 'potassium' THEN IF(raw_value BETWEEN 1.0 AND 12.5, raw_value, NULL)\n      WHEN 'sodium' THEN IF(raw_value BETWEEN 80.0 AND 200.0, raw_value, NULL)\n      WHEN 'chloride' THEN IF(raw_value BETWEEN 50.0 AND 170.0, raw_value, NULL)\n      WHEN 'glucose_serum' THEN IF(raw_value BETWEEN 10.0 AND 1800.0, raw_value, NULL)\n      WHEN 'glucose_bedside' THEN IF(raw_value BETWEEN 10.0 AND 700.0, raw_value, NULL)\n      WHEN 'calcium' THEN IF(raw_value BETWEEN 2.0 AND 20.0, raw_value, NULL)\n      WHEN 'magnesium' THEN IF(raw_value BETWEEN 0.3 AND 10.0, raw_value, NULL)\n      WHEN 'phosphate' THEN IF(raw_value BETWEEN 0.1 AND 20.0, raw_value, NULL)\n      WHEN 'hgb' THEN IF(raw_value BETWEEN 2.0 AND 25.0, raw_value, NULL)\n      WHEN 'hct' THEN IF(raw_value BETWEEN 5.0 AND 75.0, raw_value, NULL)\n      WHEN 'platelets' THEN IF(raw_value BETWEEN 1.0 AND 1500.0, raw_value, NULL)\n      WHEN 'wbc' THEN IF(raw_value BETWEEN 0.1 AND 200.0, raw_value, NULL)\n      WHEN 'albumin' THEN IF(raw_value BETWEEN 0.5 AND 7.0, raw_value, NULL)\n      WHEN 'bilirubin_total' THEN IF(raw_value BETWEEN 0.0 AND 70.0, raw_value, NULL)\n      WHEN 'ast' THEN IF(raw_value BETWEEN 1.0 AND 50000.0, raw_value, NULL)\n      WHEN 'alt' THEN IF(raw_value BETWEEN 1.0 AND 50000.0, raw_value, NULL)\n      WHEN 'inr' THEN IF(raw_value BETWEEN 0.5 AND 20.0, raw_value, NULL)\n      WHEN 'lactate' THEN IF(raw_value BETWEEN 0.1 AND 30.0, raw_value, NULL)\n      WHEN 'anion_gap' THEN IF(raw_value BETWEEN -10.0 AND 60.0, raw_value, NULL)\n      ELSE NULL\n    END AS value\n  FROM mapped\n  WHERE feature IS NOT NULL\n),\nvalid AS (\n  SELECT * FROM clean WHERE value IS NOT NULL\n),\ngrouped AS (\n  SELECT\n    patientUnitStayID,\n    feature,\n    ARRAY_AGG(value ORDER BY offset_min, labID LIMIT 1)[OFFSET(0)] AS first_value,\n    ARRAY_AGG(value ORDER BY offset_min DESC, labID DESC LIMIT 1)[OFFSET(0)] AS last_value,\n    MIN(value) AS min_value,\n    MAX(value) AS max_value,\n    AVG(value) AS mean_value,\n    COUNT(*) AS n_values\n  FROM valid\n  GROUP BY patientUnitStayID, feature\n)\nSELECT\n  patientUnitStayID,\n  MAX(IF(feature = 'creatinine', first_value, NULL)) AS x_lab_creatinine_first,\n  MAX(IF(feature = 'creatinine', last_value, NULL)) AS x_lab_creatinine_last,\n  MAX(IF(feature = 'creatinine', min_value, NULL)) AS x_lab_creatinine_min,\n  MAX(IF(feature = 'creatinine', max_value, NULL)) AS x_lab_creatinine_max,\n  MAX(IF(feature = 'creatinine', mean_value, NULL)) AS x_lab_creatinine_mean,\n  MAX(IF(feature = 'creatinine', n_values, NULL)) AS x_lab_creatinine_n,\n  MAX(IF(feature = 'creatinine' AND n_values >= 2, last_value - first_value, NULL)) AS x_lab_creatinine_delta,\n  MAX(IF(feature = 'bun', first_value, NULL)) AS x_lab_bun_first,\n  MAX(IF(feature = 'bun', last_value, NULL)) AS x_lab_bun_last,\n  MAX(IF(feature = 'bun', min_value, NULL)) AS x_lab_bun_min,\n  MAX(IF(feature = 'bun', max_value, NULL)) AS x_lab_bun_max,\n  MAX(IF(feature = 'bun', mean_value, NULL)) AS x_lab_bun_mean,\n  MAX(IF(feature = 'bun', n_values, NULL)) AS x_lab_bun_n,\n  MAX(IF(feature = 'bun' AND n_values >= 2, last_value - first_value, NULL)) AS x_lab_bun_delta,\n  MAX(IF(feature = 'bicarbonate', first_value, NULL)) AS x_lab_bicarbonate_first,\n  MAX(IF(feature = 'bicarbonate', last_value, NULL)) AS x_lab_bicarbonate_last,\n  MAX(IF(feature = 'bicarbonate', min_value, NULL)) AS x_lab_bicarbonate_min,\n  MAX(IF(feature = 'bicarbonate', max_value, NULL)) AS x_lab_bicarbonate_max,\n  MAX(IF(feature = 'bicarbonate', mean_value, NULL)) AS x_lab_bicarbonate_mean,\n  MAX(IF(feature = 'bicarbonate', n_values, NULL)) AS x_lab_bicarbonate_n,\n  MAX(IF(feature = 'bicarbonate' AND n_values >= 2, last_value - first_value, NULL)) AS x_lab_bicarbonate_delta,\n  MAX(IF(feature = 'potassium', first_value, NULL)) AS x_lab_potassium_first,\n  MAX(IF(feature = 'potassium', last_value, NULL)) AS x_lab_potassium_last,\n  MAX(IF(feature = 'potassium', min_value, NULL)) AS x_lab_potassium_min,\n  MAX(IF(feature = 'potassium', max_value, NULL)) AS x_lab_potassium_max,\n  MAX(IF(feature = 'potassium', mean_value, NULL)) AS x_lab_potassium_mean,\n  MAX(IF(feature = 'potassium', n_values, NULL)) AS x_lab_potassium_n,\n  MAX(IF(feature = 'potassium' AND n_values >= 2, last_value - first_value, NULL)) AS x_lab_potassium_delta,\n  MAX(IF(feature = 'sodium', first_value, NULL)) AS x_lab_sodium_first,\n  MAX(IF(feature = 'sodium', last_value, NULL)) AS x_lab_sodium_last,\n  MAX(IF(feature = 'sodium', min_value, NULL)) AS x_lab_sodium_min,\n  MAX(IF(feature = 'sodium', max_value, NULL)) AS x_lab_sodium_max,\n  MAX(IF(feature = 'sodium', mean_value, NULL)) AS x_lab_sodium_mean,\n  MAX(IF(feature = 'sodium', n_values, NULL)) AS x_lab_sodium_n,\n  MAX(IF(feature = 'sodium' AND n_values >= 2, last_value - first_value, NULL)) AS x_lab_sodium_delta,\n  MAX(IF(feature = 'chloride', first_value, NULL)) AS x_lab_chloride_first,\n  MAX(IF(feature = 'chloride', last_value, NULL)) AS x_lab_chloride_last,\n  MAX(IF(feature = 'chloride', min_value, NULL)) AS x_lab_chloride_min,\n  MAX(IF(feature = 'chloride', max_value, NULL)) AS x_lab_chloride_max,\n  MAX(IF(feature = 'chloride', mean_value, NULL)) AS x_lab_chloride_mean,\n  MAX(IF(feature = 'chloride', n_values, NULL)) AS x_lab_chloride_n,\n  MAX(IF(feature = 'glucose_serum', first_value, NULL)) AS x_lab_glucose_serum_first,\n  MAX(IF(feature = 'glucose_serum', last_value, NULL)) AS x_lab_glucose_serum_last,\n  MAX(IF(feature = 'glucose_serum', min_value, NULL)) AS x_lab_glucose_serum_min,\n  MAX(IF(feature = 'glucose_serum', max_value, NULL)) AS x_lab_glucose_serum_max,\n  MAX(IF(feature = 'glucose_serum', mean_value, NULL)) AS x_lab_glucose_serum_mean,\n  MAX(IF(feature = 'glucose_serum', n_values, NULL)) AS x_lab_glucose_serum_n,\n  MAX(IF(feature = 'glucose_serum' AND n_values >= 2, last_value - first_value, NULL)) AS x_lab_glucose_serum_delta,\n  MAX(IF(feature = 'glucose_bedside', first_value, NULL)) AS x_lab_glucose_bedside_first,\n  MAX(IF(feature = 'glucose_bedside', last_value, NULL)) AS x_lab_glucose_bedside_last,\n  MAX(IF(feature = 'glucose_bedside', min_value, NULL)) AS x_lab_glucose_bedside_min,\n  MAX(IF(feature = 'glucose_bedside', max_value, NULL)) AS x_lab_glucose_bedside_max,\n  MAX(IF(feature = 'glucose_bedside', mean_value, NULL)) AS x_lab_glucose_bedside_mean,\n  MAX(IF(feature = 'glucose_bedside', n_values, NULL)) AS x_lab_glucose_bedside_n,\n  MAX(IF(feature = 'calcium', first_value, NULL)) AS x_lab_calcium_first,\n  MAX(IF(feature = 'calcium', last_value, NULL)) AS x_lab_calcium_last,\n  MAX(IF(feature = 'calcium', min_value, NULL)) AS x_lab_calcium_min,\n  MAX(IF(feature = 'calcium', max_value, NULL)) AS x_lab_calcium_max,\n  MAX(IF(feature = 'calcium', mean_value, NULL)) AS x_lab_calcium_mean,\n  MAX(IF(feature = 'calcium', n_values, NULL)) AS x_lab_calcium_n,\n  MAX(IF(feature = 'magnesium', first_value, NULL)) AS x_lab_magnesium_first,\n  MAX(IF(feature = 'magnesium', last_value, NULL)) AS x_lab_magnesium_last,\n  MAX(IF(feature = 'magnesium', min_value, NULL)) AS x_lab_magnesium_min,\n  MAX(IF(feature = 'magnesium', max_value, NULL)) AS x_lab_magnesium_max,\n  MAX(IF(feature = 'magnesium', mean_value, NULL)) AS x_lab_magnesium_mean,\n  MAX(IF(feature = 'magnesium', n_values, NULL)) AS x_lab_magnesium_n,\n  MAX(IF(feature = 'phosphate', first_value, NULL)) AS x_lab_phosphate_first,\n  MAX(IF(feature = 'phosphate', last_value, NULL)) AS x_lab_phosphate_last,\n  MAX(IF(feature = 'phosphate', min_value, NULL)) AS x_lab_phosphate_min,\n  MAX(IF(feature = 'phosphate', max_value, NULL)) AS x_lab_phosphate_max,\n  MAX(IF(feature = 'phosphate', mean_value, NULL)) AS x_lab_phosphate_mean,\n  MAX(IF(feature = 'phosphate', n_values, NULL)) AS x_lab_phosphate_n,\n  MAX(IF(feature = 'hgb', first_value, NULL)) AS x_lab_hgb_first,\n  MAX(IF(feature = 'hgb', last_value, NULL)) AS x_lab_hgb_last,\n  MAX(IF(feature = 'hgb', min_value, NULL)) AS x_lab_hgb_min,\n  MAX(IF(feature = 'hgb', max_value, NULL)) AS x_lab_hgb_max,\n  MAX(IF(feature = 'hgb', mean_value, NULL)) AS x_lab_hgb_mean,\n  MAX(IF(feature = 'hgb', n_values, NULL)) AS x_lab_hgb_n,\n  MAX(IF(feature = 'hct', first_value, NULL)) AS x_lab_hct_first,\n  MAX(IF(feature = 'hct', last_value, NULL)) AS x_lab_hct_last,\n  MAX(IF(feature = 'hct', min_value, NULL)) AS x_lab_hct_min,\n  MAX(IF(feature = 'hct', max_value, NULL)) AS x_lab_hct_max,\n  MAX(IF(feature = 'hct', mean_value, NULL)) AS x_lab_hct_mean,\n  MAX(IF(feature = 'hct', n_values, NULL)) AS x_lab_hct_n,\n  MAX(IF(feature = 'platelets', first_value, NULL)) AS x_lab_platelets_first,\n  MAX(IF(feature = 'platelets', last_value, NULL)) AS x_lab_platelets_last,\n  MAX(IF(feature = 'platelets', min_value, NULL)) AS x_lab_platelets_min,\n  MAX(IF(feature = 'platelets', max_value, NULL)) AS x_lab_platelets_max,\n  MAX(IF(feature = 'platelets', mean_value, NULL)) AS x_lab_platelets_mean,\n  MAX(IF(feature = 'platelets', n_values, NULL)) AS x_lab_platelets_n,\n  MAX(IF(feature = 'wbc', first_value, NULL)) AS x_lab_wbc_first,\n  MAX(IF(feature = 'wbc', last_value, NULL)) AS x_lab_wbc_last,\n  MAX(IF(feature = 'wbc', min_value, NULL)) AS x_lab_wbc_min,\n  MAX(IF(feature = 'wbc', max_value, NULL)) AS x_lab_wbc_max,\n  MAX(IF(feature = 'wbc', mean_value, NULL)) AS x_lab_wbc_mean,\n  MAX(IF(feature = 'wbc', n_values, NULL)) AS x_lab_wbc_n,\n  MAX(IF(feature = 'albumin', first_value, NULL)) AS x_lab_albumin_first,\n  MAX(IF(feature = 'albumin', last_value, NULL)) AS x_lab_albumin_last,\n  MAX(IF(feature = 'albumin', min_value, NULL)) AS x_lab_albumin_min,\n  MAX(IF(feature = 'albumin', max_value, NULL)) AS x_lab_albumin_max,\n  MAX(IF(feature = 'albumin', mean_value, NULL)) AS x_lab_albumin_mean,\n  MAX(IF(feature = 'albumin', n_values, NULL)) AS x_lab_albumin_n,\n  MAX(IF(feature = 'bilirubin_total', first_value, NULL)) AS x_lab_bilirubin_total_first,\n  MAX(IF(feature = 'bilirubin_total', last_value, NULL)) AS x_lab_bilirubin_total_last,\n  MAX(IF(feature = 'bilirubin_total', min_value, NULL)) AS x_lab_bilirubin_total_min,\n  MAX(IF(feature = 'bilirubin_total', max_value, NULL)) AS x_lab_bilirubin_total_max,\n  MAX(IF(feature = 'bilirubin_total', mean_value, NULL)) AS x_lab_bilirubin_total_mean,\n  MAX(IF(feature = 'bilirubin_total', n_values, NULL)) AS x_lab_bilirubin_total_n,\n  MAX(IF(feature = 'ast', first_value, NULL)) AS x_lab_ast_first,\n  MAX(IF(feature = 'ast', last_value, NULL)) AS x_lab_ast_last,\n  MAX(IF(feature = 'ast', min_value, NULL)) AS x_lab_ast_min,\n  MAX(IF(feature = 'ast', max_value, NULL)) AS x_lab_ast_max,\n  MAX(IF(feature = 'ast', mean_value, NULL)) AS x_lab_ast_mean,\n  MAX(IF(feature = 'ast', n_values, NULL)) AS x_lab_ast_n,\n  MAX(IF(feature = 'alt', first_value, NULL)) AS x_lab_alt_first,\n  MAX(IF(feature = 'alt', last_value, NULL)) AS x_lab_alt_last,\n  MAX(IF(feature = 'alt', min_value, NULL)) AS x_lab_alt_min,\n  MAX(IF(feature = 'alt', max_value, NULL)) AS x_lab_alt_max,\n  MAX(IF(feature = 'alt', mean_value, NULL)) AS x_lab_alt_mean,\n  MAX(IF(feature = 'alt', n_values, NULL)) AS x_lab_alt_n,\n  MAX(IF(feature = 'inr', first_value, NULL)) AS x_lab_inr_first,\n  MAX(IF(feature = 'inr', last_value, NULL)) AS x_lab_inr_last,\n  MAX(IF(feature = 'inr', min_value, NULL)) AS x_lab_inr_min,\n  MAX(IF(feature = 'inr', max_value, NULL)) AS x_lab_inr_max,\n  MAX(IF(feature = 'inr', mean_value, NULL)) AS x_lab_inr_mean,\n  MAX(IF(feature = 'inr', n_values, NULL)) AS x_lab_inr_n,\n  MAX(IF(feature = 'lactate', first_value, NULL)) AS x_lab_lactate_first,\n  MAX(IF(feature = 'lactate', last_value, NULL)) AS x_lab_lactate_last,\n  MAX(IF(feature = 'lactate', min_value, NULL)) AS x_lab_lactate_min,\n  MAX(IF(feature = 'lactate', max_value, NULL)) AS x_lab_lactate_max,\n  MAX(IF(feature = 'lactate', mean_value, NULL)) AS x_lab_lactate_mean,\n  MAX(IF(feature = 'lactate', n_values, NULL)) AS x_lab_lactate_n,\n  MAX(IF(feature = 'lactate' AND n_values >= 2, last_value - first_value, NULL)) AS x_lab_lactate_delta,\n  MAX(IF(feature = 'anion_gap', first_value, NULL)) AS x_lab_anion_gap_first,\n  MAX(IF(feature = 'anion_gap', last_value, NULL)) AS x_lab_anion_gap_last,\n  MAX(IF(feature = 'anion_gap', min_value, NULL)) AS x_lab_anion_gap_min,\n  MAX(IF(feature = 'anion_gap', max_value, NULL)) AS x_lab_anion_gap_max,\n  MAX(IF(feature = 'anion_gap', mean_value, NULL)) AS x_lab_anion_gap_mean,\n  MAX(IF(feature = 'anion_gap', n_values, NULL)) AS x_lab_anion_gap_n\nFROM grouped\nGROUP BY patientUnitStayID;\n"
run_ddl('02_create_lab_features_v1.sql', SQL_02)

In [ ]:
SQL_02A_AUDIT = f"""
WITH main AS (
  SELECT patientUnitStayID
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
    AND deterministic_eligible_patient_stay_rank = 1
),
labs AS (
  SELECT *
  FROM `{TARGET_DATASET}.feature_labs_v1`
)
SELECT
  (SELECT COUNT(*) FROM main) AS main_cohort_rows,
  (SELECT COUNT(*) FROM labs) AS lab_table_rows,
  (SELECT COUNT(DISTINCT patientUnitStayID) FROM labs)
    AS distinct_unit_stays,
  (
    SELECT COUNT(*) - COUNT(DISTINCT patientUnitStayID)
    FROM labs
  ) AS duplicate_stay_rows,
  (
    SELECT COUNT(*)
    FROM main m
    LEFT JOIN labs l USING (patientUnitStayID)
    WHERE l.patientUnitStayID IS NULL
  ) AS patients_without_selected_labs,
  SAFE_DIVIDE(
    (SELECT COUNT(*) FROM labs),
    (SELECT COUNT(*) FROM main)
  ) AS any_selected_lab_coverage
"""

result_02A = run_aggregate(
    "02A_lab_feature_integrity_audit.sql",
    SQL_02A_AUDIT,
    "02A_lab_feature_integrity_audit.csv"
)

In [ ]:
SQL_02B_COVERAGE = f"""
WITH main AS (
  SELECT
    patientUnitStayID
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
    AND deterministic_eligible_patient_stay_rank = 1
),

joined AS (
  SELECT
    m.patientUnitStayID,
    l.* EXCEPT (patientUnitStayID)
  FROM main AS m
  LEFT JOIN `{TARGET_DATASET}.feature_labs_v1` AS l
    ON l.patientUnitStayID = m.patientUnitStayID
),

long AS (
  SELECT
    j.patientUnitStayID,
    item.feature,
    item.value
  FROM joined AS j
  CROSS JOIN UNNEST([
    STRUCT(
      'creatinine' AS feature,
      CAST(j.x_lab_creatinine_first AS FLOAT64) AS value
    ),
    STRUCT(
      'bun' AS feature,
      CAST(j.x_lab_bun_first AS FLOAT64) AS value
    ),
    STRUCT(
      'bicarbonate' AS feature,
      CAST(j.x_lab_bicarbonate_first AS FLOAT64) AS value
    ),
    STRUCT(
      'potassium' AS feature,
      CAST(j.x_lab_potassium_first AS FLOAT64) AS value
    ),
    STRUCT(
      'sodium' AS feature,
      CAST(j.x_lab_sodium_first AS FLOAT64) AS value
    ),
    STRUCT(
      'chloride' AS feature,
      CAST(j.x_lab_chloride_first AS FLOAT64) AS value
    ),
    STRUCT(
      'glucose_serum' AS feature,
      CAST(j.x_lab_glucose_serum_first AS FLOAT64) AS value
    ),
    STRUCT(
      'glucose_bedside' AS feature,
      CAST(j.x_lab_glucose_bedside_first AS FLOAT64) AS value
    ),
    STRUCT(
      'calcium' AS feature,
      CAST(j.x_lab_calcium_first AS FLOAT64) AS value
    ),
    STRUCT(
      'magnesium' AS feature,
      CAST(j.x_lab_magnesium_first AS FLOAT64) AS value
    ),
    STRUCT(
      'phosphate' AS feature,
      CAST(j.x_lab_phosphate_first AS FLOAT64) AS value
    ),
    STRUCT(
      'hgb' AS feature,
      CAST(j.x_lab_hgb_first AS FLOAT64) AS value
    ),
    STRUCT(
      'hct' AS feature,
      CAST(j.x_lab_hct_first AS FLOAT64) AS value
    ),
    STRUCT(
      'platelets' AS feature,
      CAST(j.x_lab_platelets_first AS FLOAT64) AS value
    ),
    STRUCT(
      'wbc' AS feature,
      CAST(j.x_lab_wbc_first AS FLOAT64) AS value
    ),
    STRUCT(
      'albumin' AS feature,
      CAST(j.x_lab_albumin_first AS FLOAT64) AS value
    ),
    STRUCT(
      'bilirubin_total' AS feature,
      CAST(j.x_lab_bilirubin_total_first AS FLOAT64) AS value
    ),
    STRUCT(
      'ast' AS feature,
      CAST(j.x_lab_ast_first AS FLOAT64) AS value
    ),
    STRUCT(
      'alt' AS feature,
      CAST(j.x_lab_alt_first AS FLOAT64) AS value
    ),
    STRUCT(
      'inr' AS feature,
      CAST(j.x_lab_inr_first AS FLOAT64) AS value
    ),
    STRUCT(
      'lactate' AS feature,
      CAST(j.x_lab_lactate_first AS FLOAT64) AS value
    ),
    STRUCT(
      'anion_gap' AS feature,
      CAST(j.x_lab_anion_gap_first AS FLOAT64) AS value
    )
  ]) AS item
)

SELECT
  feature,
  COUNTIF(value IS NOT NULL) AS patient_stays,
  COUNT(*) AS main_cohort,
  SAFE_DIVIDE(
    COUNTIF(value IS NOT NULL),
    COUNT(*)
  ) AS coverage
FROM long
GROUP BY feature
ORDER BY coverage DESC, feature;
"""

result_02B = run_aggregate(
    "02B_lab_feature_coverage_audit.sql",
    SQL_02B_COVERAGE,
    "02B_lab_feature_coverage_audit.csv"
)

## 03 Create Vital Features V1

In [ ]:
SQL_03 = "-- First-12-hour vital-sign summaries after broad plausibility QC.\nCREATE OR REPLACE TABLE `{{TARGET_DATASET}}.feature_vitals_v1` AS\nWITH main AS (\n  SELECT patientUnitStayID\n  FROM `{{TARGET_DATASET}}.cohort_outcome_v1`\n  WHERE eligible_main_cohort = 1\n    AND deterministic_eligible_patient_stay_rank = 1\n),\nlong_raw AS (\n  SELECT v.patientUnitStayID, v.vitalPeriodicID AS record_id, v.observationOffset AS offset_min,\n         'heart_rate' AS feature, SAFE_CAST(v.heartRate AS FLOAT64) AS raw_value\n  FROM main m JOIN `{{SOURCE_DATASET}}.vitalperiodic` v USING(patientUnitStayID)\n  WHERE v.observationOffset BETWEEN 0 AND 720\n  UNION ALL\n  SELECT v.patientUnitStayID, v.vitalPeriodicID, v.observationOffset,\n         'sao2', SAFE_CAST(v.saO2 AS FLOAT64)\n  FROM main m JOIN `{{SOURCE_DATASET}}.vitalperiodic` v USING(patientUnitStayID)\n  WHERE v.observationOffset BETWEEN 0 AND 720\n  UNION ALL\n  SELECT v.patientUnitStayID, v.vitalPeriodicID, v.observationOffset,\n         'respiratory_rate', SAFE_CAST(v.respiration AS FLOAT64)\n  FROM main m JOIN `{{SOURCE_DATASET}}.vitalperiodic` v USING(patientUnitStayID)\n  WHERE v.observationOffset BETWEEN 0 AND 720\n  UNION ALL\n  SELECT v.patientUnitStayID, v.vitalPeriodicID, v.observationOffset,\n         'temperature_c', SAFE_CAST(v.temperature AS FLOAT64)\n  FROM main m JOIN `{{SOURCE_DATASET}}.vitalperiodic` v USING(patientUnitStayID)\n  WHERE v.observationOffset BETWEEN 0 AND 720\n  UNION ALL\n  SELECT v.patientUnitStayID, v.vitalPeriodicID, v.observationOffset,\n         'invasive_systolic_bp', SAFE_CAST(v.systemicSystolic AS FLOAT64)\n  FROM main m JOIN `{{SOURCE_DATASET}}.vitalperiodic` v USING(patientUnitStayID)\n  WHERE v.observationOffset BETWEEN 0 AND 720\n  UNION ALL\n  SELECT v.patientUnitStayID, v.vitalPeriodicID, v.observationOffset,\n         'invasive_diastolic_bp', SAFE_CAST(v.systemicDiastolic AS FLOAT64)\n  FROM main m JOIN `{{SOURCE_DATASET}}.vitalperiodic` v USING(patientUnitStayID)\n  WHERE v.observationOffset BETWEEN 0 AND 720\n  UNION ALL\n  SELECT v.patientUnitStayID, v.vitalPeriodicID, v.observationOffset,\n         'invasive_mean_bp', SAFE_CAST(v.systemicMean AS FLOAT64)\n  FROM main m JOIN `{{SOURCE_DATASET}}.vitalperiodic` v USING(patientUnitStayID)\n  WHERE v.observationOffset BETWEEN 0 AND 720\n  UNION ALL\n  SELECT v.patientUnitStayID, v.vitalAperiodicID, v.observationOffset,\n         'noninvasive_systolic_bp', SAFE_CAST(v.nonInvasiveSystolic AS FLOAT64)\n  FROM main m JOIN `{{SOURCE_DATASET}}.vitalaperiodic` v USING(patientUnitStayID)\n  WHERE v.observationOffset BETWEEN 0 AND 720\n  UNION ALL\n  SELECT v.patientUnitStayID, v.vitalAperiodicID, v.observationOffset,\n         'noninvasive_diastolic_bp', SAFE_CAST(v.nonInvasiveDiastolic AS FLOAT64)\n  FROM main m JOIN `{{SOURCE_DATASET}}.vitalaperiodic` v USING(patientUnitStayID)\n  WHERE v.observationOffset BETWEEN 0 AND 720\n  UNION ALL\n  SELECT v.patientUnitStayID, v.vitalAperiodicID, v.observationOffset,\n         'noninvasive_mean_bp', SAFE_CAST(v.nonInvasiveMean AS FLOAT64)\n  FROM main m JOIN `{{SOURCE_DATASET}}.vitalaperiodic` v USING(patientUnitStayID)\n  WHERE v.observationOffset BETWEEN 0 AND 720\n),\nclean AS (\n  SELECT *,\n    CASE feature\n      WHEN 'heart_rate' THEN IF(raw_value BETWEEN 20 AND 250, raw_value, NULL)\n      WHEN 'sao2' THEN IF(raw_value BETWEEN 30 AND 100, raw_value, NULL)\n      WHEN 'respiratory_rate' THEN IF(raw_value BETWEEN 2 AND 80, raw_value, NULL)\n      WHEN 'temperature_c' THEN IF(raw_value BETWEEN 25 AND 45, raw_value, NULL)\n      WHEN 'invasive_systolic_bp' THEN IF(raw_value BETWEEN 30 AND 300, raw_value, NULL)\n      WHEN 'invasive_diastolic_bp' THEN IF(raw_value BETWEEN 10 AND 200, raw_value, NULL)\n      WHEN 'invasive_mean_bp' THEN IF(raw_value BETWEEN 20 AND 250, raw_value, NULL)\n      WHEN 'noninvasive_systolic_bp' THEN IF(raw_value BETWEEN 30 AND 300, raw_value, NULL)\n      WHEN 'noninvasive_diastolic_bp' THEN IF(raw_value BETWEEN 10 AND 200, raw_value, NULL)\n      WHEN 'noninvasive_mean_bp' THEN IF(raw_value BETWEEN 20 AND 250, raw_value, NULL)\n    END AS value\n  FROM long_raw\n),\nvalid AS (\n  SELECT * FROM clean WHERE value IS NOT NULL\n),\ngrouped AS (\n  SELECT\n    patientUnitStayID,\n    feature,\n    ARRAY_AGG(value ORDER BY offset_min, record_id LIMIT 1)[OFFSET(0)] AS first_value,\n    ARRAY_AGG(value ORDER BY offset_min DESC, record_id DESC LIMIT 1)[OFFSET(0)] AS last_value,\n    MIN(value) AS min_value,\n    MAX(value) AS max_value,\n    AVG(value) AS mean_value,\n    STDDEV_SAMP(value) AS sd_value,\n    COUNT(*) AS n_values\n  FROM valid\n  GROUP BY patientUnitStayID, feature\n)\nSELECT\n  patientUnitStayID,\n  MAX(IF(feature = 'heart_rate', first_value, NULL)) AS x_vital_heart_rate_first,\n  MAX(IF(feature = 'heart_rate', last_value, NULL)) AS x_vital_heart_rate_last,\n  MAX(IF(feature = 'heart_rate', min_value, NULL)) AS x_vital_heart_rate_min,\n  MAX(IF(feature = 'heart_rate', max_value, NULL)) AS x_vital_heart_rate_max,\n  MAX(IF(feature = 'heart_rate', mean_value, NULL)) AS x_vital_heart_rate_mean,\n  MAX(IF(feature = 'heart_rate', sd_value, NULL)) AS x_vital_heart_rate_sd,\n  MAX(IF(feature = 'heart_rate', n_values, NULL)) AS x_vital_heart_rate_n,\n  MAX(IF(feature = 'sao2', first_value, NULL)) AS x_vital_sao2_first,\n  MAX(IF(feature = 'sao2', last_value, NULL)) AS x_vital_sao2_last,\n  MAX(IF(feature = 'sao2', min_value, NULL)) AS x_vital_sao2_min,\n  MAX(IF(feature = 'sao2', max_value, NULL)) AS x_vital_sao2_max,\n  MAX(IF(feature = 'sao2', mean_value, NULL)) AS x_vital_sao2_mean,\n  MAX(IF(feature = 'sao2', sd_value, NULL)) AS x_vital_sao2_sd,\n  MAX(IF(feature = 'sao2', n_values, NULL)) AS x_vital_sao2_n,\n  MAX(IF(feature = 'respiratory_rate', first_value, NULL)) AS x_vital_respiratory_rate_first,\n  MAX(IF(feature = 'respiratory_rate', last_value, NULL)) AS x_vital_respiratory_rate_last,\n  MAX(IF(feature = 'respiratory_rate', min_value, NULL)) AS x_vital_respiratory_rate_min,\n  MAX(IF(feature = 'respiratory_rate', max_value, NULL)) AS x_vital_respiratory_rate_max,\n  MAX(IF(feature = 'respiratory_rate', mean_value, NULL)) AS x_vital_respiratory_rate_mean,\n  MAX(IF(feature = 'respiratory_rate', sd_value, NULL)) AS x_vital_respiratory_rate_sd,\n  MAX(IF(feature = 'respiratory_rate', n_values, NULL)) AS x_vital_respiratory_rate_n,\n  MAX(IF(feature = 'temperature_c', first_value, NULL)) AS x_vital_temperature_c_first,\n  MAX(IF(feature = 'temperature_c', last_value, NULL)) AS x_vital_temperature_c_last,\n  MAX(IF(feature = 'temperature_c', min_value, NULL)) AS x_vital_temperature_c_min,\n  MAX(IF(feature = 'temperature_c', max_value, NULL)) AS x_vital_temperature_c_max,\n  MAX(IF(feature = 'temperature_c', mean_value, NULL)) AS x_vital_temperature_c_mean,\n  MAX(IF(feature = 'temperature_c', sd_value, NULL)) AS x_vital_temperature_c_sd,\n  MAX(IF(feature = 'temperature_c', n_values, NULL)) AS x_vital_temperature_c_n,\n  MAX(IF(feature = 'invasive_systolic_bp', first_value, NULL)) AS x_vital_invasive_systolic_bp_first,\n  MAX(IF(feature = 'invasive_systolic_bp', last_value, NULL)) AS x_vital_invasive_systolic_bp_last,\n  MAX(IF(feature = 'invasive_systolic_bp', min_value, NULL)) AS x_vital_invasive_systolic_bp_min,\n  MAX(IF(feature = 'invasive_systolic_bp', max_value, NULL)) AS x_vital_invasive_systolic_bp_max,\n  MAX(IF(feature = 'invasive_systolic_bp', mean_value, NULL)) AS x_vital_invasive_systolic_bp_mean,\n  MAX(IF(feature = 'invasive_systolic_bp', sd_value, NULL)) AS x_vital_invasive_systolic_bp_sd,\n  MAX(IF(feature = 'invasive_systolic_bp', n_values, NULL)) AS x_vital_invasive_systolic_bp_n,\n  MAX(IF(feature = 'invasive_diastolic_bp', first_value, NULL)) AS x_vital_invasive_diastolic_bp_first,\n  MAX(IF(feature = 'invasive_diastolic_bp', last_value, NULL)) AS x_vital_invasive_diastolic_bp_last,\n  MAX(IF(feature = 'invasive_diastolic_bp', min_value, NULL)) AS x_vital_invasive_diastolic_bp_min,\n  MAX(IF(feature = 'invasive_diastolic_bp', max_value, NULL)) AS x_vital_invasive_diastolic_bp_max,\n  MAX(IF(feature = 'invasive_diastolic_bp', mean_value, NULL)) AS x_vital_invasive_diastolic_bp_mean,\n  MAX(IF(feature = 'invasive_diastolic_bp', sd_value, NULL)) AS x_vital_invasive_diastolic_bp_sd,\n  MAX(IF(feature = 'invasive_diastolic_bp', n_values, NULL)) AS x_vital_invasive_diastolic_bp_n,\n  MAX(IF(feature = 'invasive_mean_bp', first_value, NULL)) AS x_vital_invasive_mean_bp_first,\n  MAX(IF(feature = 'invasive_mean_bp', last_value, NULL)) AS x_vital_invasive_mean_bp_last,\n  MAX(IF(feature = 'invasive_mean_bp', min_value, NULL)) AS x_vital_invasive_mean_bp_min,\n  MAX(IF(feature = 'invasive_mean_bp', max_value, NULL)) AS x_vital_invasive_mean_bp_max,\n  MAX(IF(feature = 'invasive_mean_bp', mean_value, NULL)) AS x_vital_invasive_mean_bp_mean,\n  MAX(IF(feature = 'invasive_mean_bp', sd_value, NULL)) AS x_vital_invasive_mean_bp_sd,\n  MAX(IF(feature = 'invasive_mean_bp', n_values, NULL)) AS x_vital_invasive_mean_bp_n,\n  MAX(IF(feature = 'noninvasive_systolic_bp', first_value, NULL)) AS x_vital_noninvasive_systolic_bp_first,\n  MAX(IF(feature = 'noninvasive_systolic_bp', last_value, NULL)) AS x_vital_noninvasive_systolic_bp_last,\n  MAX(IF(feature = 'noninvasive_systolic_bp', min_value, NULL)) AS x_vital_noninvasive_systolic_bp_min,\n  MAX(IF(feature = 'noninvasive_systolic_bp', max_value, NULL)) AS x_vital_noninvasive_systolic_bp_max,\n  MAX(IF(feature = 'noninvasive_systolic_bp', mean_value, NULL)) AS x_vital_noninvasive_systolic_bp_mean,\n  MAX(IF(feature = 'noninvasive_systolic_bp', sd_value, NULL)) AS x_vital_noninvasive_systolic_bp_sd,\n  MAX(IF(feature = 'noninvasive_systolic_bp', n_values, NULL)) AS x_vital_noninvasive_systolic_bp_n,\n  MAX(IF(feature = 'noninvasive_diastolic_bp', first_value, NULL)) AS x_vital_noninvasive_diastolic_bp_first,\n  MAX(IF(feature = 'noninvasive_diastolic_bp', last_value, NULL)) AS x_vital_noninvasive_diastolic_bp_last,\n  MAX(IF(feature = 'noninvasive_diastolic_bp', min_value, NULL)) AS x_vital_noninvasive_diastolic_bp_min,\n  MAX(IF(feature = 'noninvasive_diastolic_bp', max_value, NULL)) AS x_vital_noninvasive_diastolic_bp_max,\n  MAX(IF(feature = 'noninvasive_diastolic_bp', mean_value, NULL)) AS x_vital_noninvasive_diastolic_bp_mean,\n  MAX(IF(feature = 'noninvasive_diastolic_bp', sd_value, NULL)) AS x_vital_noninvasive_diastolic_bp_sd,\n  MAX(IF(feature = 'noninvasive_diastolic_bp', n_values, NULL)) AS x_vital_noninvasive_diastolic_bp_n,\n  MAX(IF(feature = 'noninvasive_mean_bp', first_value, NULL)) AS x_vital_noninvasive_mean_bp_first,\n  MAX(IF(feature = 'noninvasive_mean_bp', last_value, NULL)) AS x_vital_noninvasive_mean_bp_last,\n  MAX(IF(feature = 'noninvasive_mean_bp', min_value, NULL)) AS x_vital_noninvasive_mean_bp_min,\n  MAX(IF(feature = 'noninvasive_mean_bp', max_value, NULL)) AS x_vital_noninvasive_mean_bp_max,\n  MAX(IF(feature = 'noninvasive_mean_bp', mean_value, NULL)) AS x_vital_noninvasive_mean_bp_mean,\n  MAX(IF(feature = 'noninvasive_mean_bp', sd_value, NULL)) AS x_vital_noninvasive_mean_bp_sd,\n  MAX(IF(feature = 'noninvasive_mean_bp', n_values, NULL)) AS x_vital_noninvasive_mean_bp_n\nFROM grouped\nGROUP BY patientUnitStayID;\n"
run_ddl('03_create_vital_features_v1.sql', SQL_03)

In [ ]:
SQL_03A_AUDIT = f"""
WITH main AS (
  SELECT
    patientUnitStayID
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
    AND deterministic_eligible_patient_stay_rank = 1
),

vitals AS (
  SELECT *
  FROM `{TARGET_DATASET}.feature_vitals_v1`
)

SELECT
  (SELECT COUNT(*) FROM main) AS main_cohort_rows,

  (SELECT COUNT(*) FROM vitals) AS vital_table_rows,

  (
    SELECT COUNT(DISTINCT patientUnitStayID)
    FROM vitals
  ) AS distinct_unit_stays,

  (
    SELECT COUNT(*) - COUNT(DISTINCT patientUnitStayID)
    FROM vitals
  ) AS duplicate_stay_rows,

  (
    SELECT COUNT(*)
    FROM main AS m
    LEFT JOIN vitals AS v
      ON v.patientUnitStayID = m.patientUnitStayID
    WHERE v.patientUnitStayID IS NULL
  ) AS patients_without_selected_vitals,

  SAFE_DIVIDE(
    (SELECT COUNT(*) FROM vitals),
    (SELECT COUNT(*) FROM main)
  ) AS any_selected_vital_coverage;
"""

result_03A = run_aggregate(
    "03A_vital_feature_integrity_audit.sql",
    SQL_03A_AUDIT,
    "03A_vital_feature_integrity_audit.csv"
)

In [ ]:
SQL_03B_COVERAGE = f"""
WITH main AS (
  SELECT
    patientUnitStayID
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
    AND deterministic_eligible_patient_stay_rank = 1
),

joined AS (
  SELECT
    m.patientUnitStayID,
    v.* EXCEPT (patientUnitStayID)
  FROM main AS m
  LEFT JOIN `{TARGET_DATASET}.feature_vitals_v1` AS v
    ON v.patientUnitStayID = m.patientUnitStayID
),

long AS (
  SELECT
    j.patientUnitStayID,
    item.feature,
    item.n_values,
    item.min_value,
    item.max_value
  FROM joined AS j

  CROSS JOIN UNNEST([

    STRUCT(
      'heart_rate' AS feature,
      CAST(j.x_vital_heart_rate_n AS INT64) AS n_values,
      CAST(j.x_vital_heart_rate_min AS FLOAT64) AS min_value,
      CAST(j.x_vital_heart_rate_max AS FLOAT64) AS max_value
    ),

    STRUCT(
      'sao2' AS feature,
      CAST(j.x_vital_sao2_n AS INT64) AS n_values,
      CAST(j.x_vital_sao2_min AS FLOAT64) AS min_value,
      CAST(j.x_vital_sao2_max AS FLOAT64) AS max_value
    ),

    STRUCT(
      'respiratory_rate' AS feature,
      CAST(j.x_vital_respiratory_rate_n AS INT64) AS n_values,
      CAST(j.x_vital_respiratory_rate_min AS FLOAT64) AS min_value,
      CAST(j.x_vital_respiratory_rate_max AS FLOAT64) AS max_value
    ),

    STRUCT(
      'temperature_c' AS feature,
      CAST(j.x_vital_temperature_c_n AS INT64) AS n_values,
      CAST(j.x_vital_temperature_c_min AS FLOAT64) AS min_value,
      CAST(j.x_vital_temperature_c_max AS FLOAT64) AS max_value
    ),

    STRUCT(
      'invasive_systolic_bp' AS feature,
      CAST(j.x_vital_invasive_systolic_bp_n AS INT64) AS n_values,
      CAST(j.x_vital_invasive_systolic_bp_min AS FLOAT64) AS min_value,
      CAST(j.x_vital_invasive_systolic_bp_max AS FLOAT64) AS max_value
    ),

    STRUCT(
      'invasive_diastolic_bp' AS feature,
      CAST(j.x_vital_invasive_diastolic_bp_n AS INT64) AS n_values,
      CAST(j.x_vital_invasive_diastolic_bp_min AS FLOAT64) AS min_value,
      CAST(j.x_vital_invasive_diastolic_bp_max AS FLOAT64) AS max_value
    ),

    STRUCT(
      'invasive_mean_bp' AS feature,
      CAST(j.x_vital_invasive_mean_bp_n AS INT64) AS n_values,
      CAST(j.x_vital_invasive_mean_bp_min AS FLOAT64) AS min_value,
      CAST(j.x_vital_invasive_mean_bp_max AS FLOAT64) AS max_value
    ),

    STRUCT(
      'noninvasive_systolic_bp' AS feature,
      CAST(j.x_vital_noninvasive_systolic_bp_n AS INT64) AS n_values,
      CAST(j.x_vital_noninvasive_systolic_bp_min AS FLOAT64) AS min_value,
      CAST(j.x_vital_noninvasive_systolic_bp_max AS FLOAT64) AS max_value
    ),

    STRUCT(
      'noninvasive_diastolic_bp' AS feature,
      CAST(j.x_vital_noninvasive_diastolic_bp_n AS INT64) AS n_values,
      CAST(j.x_vital_noninvasive_diastolic_bp_min AS FLOAT64) AS min_value,
      CAST(j.x_vital_noninvasive_diastolic_bp_max AS FLOAT64) AS max_value
    ),

    STRUCT(
      'noninvasive_mean_bp' AS feature,
      CAST(j.x_vital_noninvasive_mean_bp_n AS INT64) AS n_values,
      CAST(j.x_vital_noninvasive_mean_bp_min AS FLOAT64) AS min_value,
      CAST(j.x_vital_noninvasive_mean_bp_max AS FLOAT64) AS max_value
    )

  ]) AS item
)

SELECT
  feature,

  COUNTIF(n_values > 0) AS patient_stays,

  COUNT(*) AS main_cohort,

  SAFE_DIVIDE(
    COUNTIF(n_values > 0),
    COUNT(*)
  ) AS coverage,

  SUM(COALESCE(n_values, 0)) AS total_valid_measurements,

  MIN(min_value) AS qc_observed_min,

  MAX(max_value) AS qc_observed_max

FROM long

GROUP BY feature

ORDER BY coverage DESC, feature;
"""

result_03B = run_aggregate(
    "03B_vital_feature_coverage_audit.sql",
    SQL_03B_COVERAGE,
    "03B_vital_feature_coverage_audit.csv"
)

## 04 Create Context Features V1

In [ ]:
SQL_04 = r"""
-- Structured first-12-hour treatment, respiratory-support
-- and past-history features.
--
-- Respiratory definitions are intentionally conservative:
-- 1. ventStartOffset alone is NOT treated as proof of invasive ventilation.
-- 2. Explicit non-invasive ventilation terms are excluded from the
--    invasive-mechanical-ventilation definition.
-- 3. Direct airway documentation and explicit treatment documentation
--    are retained separately.

CREATE OR REPLACE TABLE `{{TARGET_DATASET}}.feature_context_v1` AS

WITH main AS (
  SELECT
    patientUnitStayID
  FROM `{{TARGET_DATASET}}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
    AND deterministic_eligible_patient_stay_rank = 1
),

/* ============================================================
   PAST MEDICAL HISTORY
   ============================================================ */

ph_raw AS (
  SELECT
    p.patientUnitStayID,
    LOWER(COALESCE(p.pastHistoryPath, '')) AS path,
    LOWER(COALESCE(p.pastHistoryValue, '')) AS value
  FROM main AS m
  INNER JOIN `{{SOURCE_DATASET}}.pasthistory` AS p
    USING (patientUnitStayID)
  WHERE p.pastHistoryOffset <= 720
),

ph AS (
  SELECT
    patientUnitStayID,

    1 AS x_hx_past_history_documented,

    MAX(
      IF(
        REGEXP_CONTAINS(
          CONCAT(path, '|', value),
          r'hypertension'
        ),
        1,
        0
      )
    ) AS x_hx_hypertension,

    MAX(
      IF(
        REGEXP_CONTAINS(
          CONCAT(path, '|', value),
          r'diabet|insulin dependent'
        )
        OR (
          REGEXP_CONTAINS(path, r'diabet')
          AND value = 'medication dependent'
        ),
        1,
        0
      )
    ) AS x_hx_diabetes,

    MAX(
      IF(
        REGEXP_CONTAINS(
          CONCAT(path, '|', value),
          r'congestive heart failure|\bchf\b'
        ),
        1,
        0
      )
    ) AS x_hx_chf,

    MAX(
      IF(
        REGEXP_CONTAINS(
          CONCAT(path, '|', value),
          r'coronary artery|myocardial infarction|\bmi\b|angina|cabg|ptca|pci'
        ),
        1,
        0
      )
    ) AS x_hx_cad,

    MAX(
      IF(
        REGEXP_CONTAINS(
          CONCAT(path, '|', value),
          r'copd|chronic obstructive|emphysema|chronic bronchitis'
        ),
        1,
        0
      )
    ) AS x_hx_chronic_pulmonary,

    MAX(
      IF(
        REGEXP_CONTAINS(
          CONCAT(path, '|', value),
          r'cirrhosis|portal hypertension|hepatic failure|chronic liver'
        ),
        1,
        0
      )
    ) AS x_hx_chronic_liver,

    MAX(
      IF(
        REGEXP_CONTAINS(
          CONCAT(path, '|', value),
          r'/cancer/|hematologic malignancy|leukemia|lymphoma|myeloma|metastases'
        ),
        1,
        0
      )
    ) AS x_hx_malignancy,

    MAX(
      IF(
        REGEXP_CONTAINS(
          CONCAT(path, '|', value),
          r'immunosupp|chemotherapy|radiation therapy|transplant|\baids\b'
        ),
        1,
        0
      )
    ) AS x_hx_immunosuppression,

    MAX(
      IF(
        REGEXP_CONTAINS(
          CONCAT(path, '|', value),
          r'stroke|cerebrovascular|\bcva\b|\btia\b'
        ),
        1,
        0
      )
    ) AS x_hx_cerebrovascular,

    MAX(
      IF(
        REGEXP_CONTAINS(
          CONCAT(path, '|', value),
          r'renal insufficiency|chronic kidney|\bckd\b'
        )
        AND NOT REGEXP_CONTAINS(
          CONCAT(path, '|', value),
          r'dialysis|esrd|end stage'
        ),
        1,
        0
      )
    ) AS x_hx_chronic_kidney_non_esrd

  FROM ph_raw
  GROUP BY patientUnitStayID
),

/* ============================================================
   INFUSION DRUGS
   ============================================================ */

inf_raw AS (
  SELECT
    i.patientUnitStayID,
    LOWER(TRIM(i.drugName)) AS drug
  FROM main AS m
  INNER JOIN `{{SOURCE_DATASET}}.infusiondrug` AS i
    USING (patientUnitStayID)
  WHERE i.infusionOffset BETWEEN 0 AND 720
    AND NULLIF(TRIM(i.drugName), '') IS NOT NULL
),

inf AS (
  SELECT
    patientUnitStayID,

    MAX(
      IF(
        REGEXP_CONTAINS(drug, r'norepine|levophed'),
        1,
        0
      )
    ) AS x_tx_norepinephrine,

    MAX(
      IF(
        REGEXP_CONTAINS(
          drug,
          r'(^|[^a-z])epinephrine([^a-z]|$)|adrenalin'
        ),
        1,
        0
      )
    ) AS x_tx_epinephrine,

    MAX(
      IF(REGEXP_CONTAINS(drug, r'vasopressin'), 1, 0)
    ) AS x_tx_vasopressin,

    MAX(
      IF(
        REGEXP_CONTAINS(
          drug,
          r'phenylephrine|neosynephrine|neo-synephrine'
        ),
        1,
        0
      )
    ) AS x_tx_phenylephrine,

    MAX(
      IF(
        REGEXP_CONTAINS(
          drug,
          r'(^|[^a-z])dopamine([^a-z]|$)'
        ),
        1,
        0
      )
    ) AS x_tx_dopamine,

    MAX(
      IF(REGEXP_CONTAINS(drug, r'dobutamine'), 1, 0)
    ) AS x_tx_dobutamine,

    MAX(
      IF(REGEXP_CONTAINS(drug, r'milrinone'), 1, 0)
    ) AS x_tx_milrinone,

    MAX(
      IF(REGEXP_CONTAINS(drug, r'propofol'), 1, 0)
    ) AS x_tx_propofol,

    MAX(
      IF(
        REGEXP_CONTAINS(drug, r'midazolam|versed'),
        1,
        0
      )
    ) AS x_tx_midazolam,

    MAX(
      IF(
        REGEXP_CONTAINS(
          drug,
          r'dexmedetomidine|precedex'
        ),
        1,
        0
      )
    ) AS x_tx_dexmedetomidine,

    MAX(
      IF(REGEXP_CONTAINS(drug, r'insulin'), 1, 0)
    ) AS x_tx_insulin_infusion,

    MAX(
      IF(
        REGEXP_CONTAINS(
          drug,
          r'furosemide|lasix|bumetanide|bumex'
        ),
        1,
        0
      )
    ) AS x_tx_loop_diuretic_infusion,

    MAX(
      IF(REGEXP_CONTAINS(drug, r'bicarbonate'), 1, 0)
    ) AS x_tx_bicarbonate_infusion,

    MAX(
      IF(REGEXP_CONTAINS(drug, r'heparin'), 1, 0)
    ) AS x_tx_heparin_infusion,

    MAX(
      IF(
        REGEXP_CONTAINS(
          drug,
          r'nicardipine|clevidipine|nitroprusside'
        ),
        1,
        0
      )
    ) AS x_tx_antihypertensive_infusion

  FROM inf_raw
  GROUP BY patientUnitStayID
),

/* ============================================================
   RESPIRATORY CARE: DIRECT AIRWAY DOCUMENTATION
   ============================================================ */

rc_raw AS (
  SELECT
    r.patientUnitStayID,
    LOWER(TRIM(COALESCE(r.airwayType, ''))) AS airway,
    r.ventStartOffset,
    r.respCareStatusOffset
  FROM main AS m
  INNER JOIN `{{SOURCE_DATASET}}.respiratorycare` AS r
    USING (patientUnitStayID)
  WHERE r.ventStartOffset BETWEEN 0 AND 720
     OR r.respCareStatusOffset BETWEEN 0 AND 720
),

rc AS (
  SELECT
    patientUnitStayID,

    MAX(
      IF(
        REGEXP_CONTAINS(
          airway,
          r'oral ett|nasal ett|endotracheal'
        ),
        1,
        0
      )
    ) AS rc_endotracheal_airway,

    MAX(
      IF(
        REGEXP_CONTAINS(airway, r'tracheostomy'),
        1,
        0
      )
    ) AS rc_tracheostomy,

    -- An invasive airway documented during the first 12 hours
    -- is considered direct evidence of invasive support.
    MAX(
      IF(
        REGEXP_CONTAINS(
          airway,
          r'oral ett|nasal ett|endotracheal|tracheostomy'
        ),
        1,
        0
      )
    ) AS rc_direct_invasive_support,

    -- Blank airway attached to an actual respiratory-care status
    -- entry in the first 12 hours. ventStartOffset alone is not used.
    MAX(
      IF(
        respCareStatusOffset BETWEEN 0 AND 720
        AND airway = '',
        1,
        0
      )
    ) AS rc_uncertain

  FROM rc_raw
  GROUP BY patientUnitStayID
),

/* ============================================================
   TREATMENT: EXPLICIT RESPIRATORY SUPPORT TERMS
   ============================================================ */

tx_raw AS (
  SELECT
    t.patientUnitStayID,
    LOWER(TRIM(t.treatmentString)) AS item
  FROM main AS m
  INNER JOIN `{{SOURCE_DATASET}}.treatment` AS t
    USING (patientUnitStayID)
  WHERE t.treatmentOffset BETWEEN 0 AND 720
    AND NULLIF(TRIM(t.treatmentString), '') IS NOT NULL
),

tx AS (
  SELECT
    patientUnitStayID,

    -- Explicit invasive ventilation terms.
    -- Records containing an explicit non-invasive designation
    -- are excluded even if they also contain "mechanical ventilation".
    MAX(
      IF(
        (
          REGEXP_CONTAINS(
            item,
            r'mechanical ventilation|invasive ventilation|intubat|ventilator weaning'
          )
          AND NOT REGEXP_CONTAINS(
            item,
            r'non.?invasive ventilation|\bbipap\b|bi-pap'
          )
        ),
        1,
        0
      )
    ) AS tx_invasive_vent,

    -- Conservative NIV definition: only explicit NIV or BiPAP terms.
    -- Generic CPAP/PEEP therapy is not automatically classified as NIV,
    -- because PEEP may also be used during invasive ventilation.
    MAX(
      IF(
        REGEXP_CONTAINS(
          item,
          r'non.?invasive ventilation|\bbipap\b|bi-pap'
        ),
        1,
        0
      )
    ) AS tx_noninvasive_vent,

    MAX(
      IF(
        REGEXP_CONTAINS(
          item,
          r'high flow|high-flow|\bhfnc\b'
        ),
        1,
        0
      )
    ) AS tx_high_flow,

    MAX(
      IF(
        REGEXP_CONTAINS(
          item,
          r'oxygen therapy|supplemental oxygen|nasal cannula|face mask|non-rebreather'
        ),
        1,
        0
      )
    ) AS tx_oxygen

  FROM tx_raw
  GROUP BY patientUnitStayID
)

/* ============================================================
   FINAL ONE-ROW-PER-PATIENT CONTEXT TABLE
   ============================================================ */

SELECT
  m.patientUnitStayID,

  COALESCE(
    ph.x_hx_past_history_documented,
    0
  ) AS x_hx_past_history_documented,

  COALESCE(ph.x_hx_hypertension, 0)
    AS x_hx_hypertension,

  COALESCE(ph.x_hx_diabetes, 0)
    AS x_hx_diabetes,

  COALESCE(ph.x_hx_chf, 0)
    AS x_hx_chf,

  COALESCE(ph.x_hx_cad, 0)
    AS x_hx_cad,

  COALESCE(ph.x_hx_chronic_pulmonary, 0)
    AS x_hx_chronic_pulmonary,

  COALESCE(ph.x_hx_chronic_liver, 0)
    AS x_hx_chronic_liver,

  COALESCE(ph.x_hx_malignancy, 0)
    AS x_hx_malignancy,

  COALESCE(ph.x_hx_immunosuppression, 0)
    AS x_hx_immunosuppression,

  COALESCE(ph.x_hx_cerebrovascular, 0)
    AS x_hx_cerebrovascular,

  COALESCE(ph.x_hx_chronic_kidney_non_esrd, 0)
    AS x_hx_chronic_kidney_non_esrd,

  IF(
    COALESCE(inf.x_tx_norepinephrine, 0)
    + COALESCE(inf.x_tx_epinephrine, 0)
    + COALESCE(inf.x_tx_vasopressin, 0)
    + COALESCE(inf.x_tx_phenylephrine, 0)
    + COALESCE(inf.x_tx_dopamine, 0) > 0,
    1,
    0
  ) AS x_tx_any_vasopressor,

  COALESCE(inf.x_tx_norepinephrine, 0)
    AS x_tx_norepinephrine,

  COALESCE(inf.x_tx_epinephrine, 0)
    AS x_tx_epinephrine,

  COALESCE(inf.x_tx_vasopressin, 0)
    AS x_tx_vasopressin,

  COALESCE(inf.x_tx_phenylephrine, 0)
    AS x_tx_phenylephrine,

  COALESCE(inf.x_tx_dopamine, 0)
    AS x_tx_dopamine,

  COALESCE(inf.x_tx_dobutamine, 0)
    AS x_tx_dobutamine,

  COALESCE(inf.x_tx_milrinone, 0)
    AS x_tx_milrinone,

  COALESCE(inf.x_tx_propofol, 0)
    AS x_tx_propofol,

  COALESCE(inf.x_tx_midazolam, 0)
    AS x_tx_midazolam,

  COALESCE(inf.x_tx_dexmedetomidine, 0)
    AS x_tx_dexmedetomidine,

  COALESCE(inf.x_tx_insulin_infusion, 0)
    AS x_tx_insulin_infusion,

  COALESCE(inf.x_tx_loop_diuretic_infusion, 0)
    AS x_tx_loop_diuretic_infusion,

  COALESCE(inf.x_tx_bicarbonate_infusion, 0)
    AS x_tx_bicarbonate_infusion,

  COALESCE(inf.x_tx_heparin_infusion, 0)
    AS x_tx_heparin_infusion,

  COALESCE(inf.x_tx_antihypertensive_infusion, 0)
    AS x_tx_antihypertensive_infusion,

  IF(
    COALESCE(rc.rc_endotracheal_airway, 0) = 1
    OR COALESCE(rc.rc_tracheostomy, 0) = 1,
    1,
    0
  ) AS x_resp_invasive_airway,

  COALESCE(rc.rc_tracheostomy, 0)
    AS x_resp_tracheostomy,

  IF(
    COALESCE(rc.rc_direct_invasive_support, 0) = 1
    OR COALESCE(tx.tx_invasive_vent, 0) = 1,
    1,
    0
  ) AS x_resp_invasive_mechanical_ventilation,

  COALESCE(tx.tx_noninvasive_vent, 0)
    AS x_resp_noninvasive_ventilation,

  COALESCE(tx.tx_high_flow, 0)
    AS x_resp_high_flow_oxygen,

  COALESCE(tx.tx_oxygen, 0)
    AS x_resp_oxygen_therapy,

  IF(
    COALESCE(rc.rc_uncertain, 0) = 1
    AND COALESCE(tx.tx_invasive_vent, 0) = 0
    AND COALESCE(tx.tx_noninvasive_vent, 0) = 0,
    1,
    0
  ) AS x_resp_respiratory_documented_uncertain

FROM main AS m

LEFT JOIN ph
  USING (patientUnitStayID)

LEFT JOIN inf
  USING (patientUnitStayID)

LEFT JOIN rc
  USING (patientUnitStayID)

LEFT JOIN tx
  USING (patientUnitStayID);
"""

run_ddl(
    "04_create_context_features_v1.sql",
    SQL_04
)

In [ ]:
SQL_04A_AUDIT = f"""
WITH main AS (
  SELECT
    patientUnitStayID
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
    AND deterministic_eligible_patient_stay_rank = 1
),

context AS (
  SELECT *
  FROM `{TARGET_DATASET}.feature_context_v1`
)

SELECT
  (SELECT COUNT(*) FROM main) AS main_cohort_rows,

  (SELECT COUNT(*) FROM context) AS context_table_rows,

  (
    SELECT COUNT(DISTINCT patientUnitStayID)
    FROM context
  ) AS distinct_unit_stays,

  (
    SELECT COUNT(*) - COUNT(DISTINCT patientUnitStayID)
    FROM context
  ) AS duplicate_stay_rows,

  (
    SELECT COUNT(*)
    FROM main AS m
    LEFT JOIN context AS c
      ON c.patientUnitStayID = m.patientUnitStayID
    WHERE c.patientUnitStayID IS NULL
  ) AS patients_missing_context_row,

  (
    SELECT COUNT(*)
    FROM context
    WHERE x_tx_any_vasopressor !=
      IF(
        x_tx_norepinephrine
        + x_tx_epinephrine
        + x_tx_vasopressin
        + x_tx_phenylephrine
        + x_tx_dopamine > 0,
        1,
        0
      )
  ) AS inconsistent_any_vasopressor,

  (
    SELECT COUNT(*)
    FROM context
    WHERE x_resp_tracheostomy = 1
      AND x_resp_invasive_airway = 0
  ) AS inconsistent_tracheostomy_airway,

  (
    SELECT COUNT(*)
    FROM context
    WHERE x_resp_invasive_mechanical_ventilation = 1
      AND x_resp_invasive_airway = 0
  ) AS invasive_vent_without_documented_airway,

  (
    SELECT COUNT(*)
    FROM context
    WHERE x_resp_invasive_airway = 1
      AND x_resp_invasive_mechanical_ventilation = 0
  ) AS invasive_airway_without_documented_ventilation

FROM context
LIMIT 1;
"""

result_04A = run_aggregate(
    "04A_context_feature_integrity_audit.sql",
    SQL_04A_AUDIT,
    "04A_context_feature_integrity_audit.csv"
)

In [ ]:
SQL_04B_PREVALENCE = f"""
WITH long AS (
  SELECT
    patientUnitStayID,
    feature,
    value
  FROM `{TARGET_DATASET}.feature_context_v1`

  UNPIVOT (
    value FOR feature IN (
      x_hx_past_history_documented,
      x_hx_hypertension,
      x_hx_diabetes,
      x_hx_chf,
      x_hx_cad,
      x_hx_chronic_pulmonary,
      x_hx_chronic_liver,
      x_hx_malignancy,
      x_hx_immunosuppression,
      x_hx_cerebrovascular,
      x_hx_chronic_kidney_non_esrd,

      x_tx_any_vasopressor,
      x_tx_norepinephrine,
      x_tx_epinephrine,
      x_tx_vasopressin,
      x_tx_phenylephrine,
      x_tx_dopamine,
      x_tx_dobutamine,
      x_tx_milrinone,
      x_tx_propofol,
      x_tx_midazolam,
      x_tx_dexmedetomidine,
      x_tx_insulin_infusion,
      x_tx_loop_diuretic_infusion,
      x_tx_bicarbonate_infusion,
      x_tx_heparin_infusion,
      x_tx_antihypertensive_infusion,

      x_resp_invasive_airway,
      x_resp_tracheostomy,
      x_resp_invasive_mechanical_ventilation,
      x_resp_noninvasive_ventilation,
      x_resp_high_flow_oxygen,
      x_resp_oxygen_therapy,
      x_resp_respiratory_documented_uncertain
    )
  )
)

SELECT
  feature,
  COUNT(*) AS cohort_rows,
  COUNTIF(value = 1) AS positive_patients,
  SAFE_DIVIDE(
    COUNTIF(value = 1),
    COUNT(*)
  ) AS prevalence,
  COUNTIF(value IS NULL) AS missing_values,
  COUNTIF(value NOT IN (0, 1)) AS nonbinary_values

FROM long
GROUP BY feature
ORDER BY feature;
"""

result_04B = run_aggregate(
    "04B_context_feature_prevalence_audit.sql",
    SQL_04B_PREVALENCE,
    "04B_context_feature_prevalence_audit.csv"
)

In [ ]:
SQL_04C_RESP_TERM_AUDIT = f"""
WITH main AS (
  SELECT
    patientUnitStayID
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
    AND deterministic_eligible_patient_stay_rank = 1
),

direct_airway AS (
  SELECT
    r.patientUnitStayID,

    MAX(
      IF(
        REGEXP_CONTAINS(
          LOWER(TRIM(COALESCE(r.airwayType, ''))),
          r'oral ett|nasal ett|endotracheal|tracheostomy'
        ),
        1,
        0
      )
    ) AS invasive_airway

  FROM main AS m

  INNER JOIN `{SOURCE_DATASET}.respiratorycare` AS r
    ON r.patientUnitStayID = m.patientUnitStayID

  WHERE
    r.ventStartOffset BETWEEN 0 AND 720
    OR r.respCareStatusOffset BETWEEN 0 AND 720

  GROUP BY r.patientUnitStayID
),

matched_treatments AS (
  SELECT
    t.patientUnitStayID,
    LOWER(TRIM(t.treatmentString)) AS treatment_item

  FROM main AS m

  INNER JOIN `{SOURCE_DATASET}.treatment` AS t
    ON t.patientUnitStayID = m.patientUnitStayID

  WHERE t.treatmentOffset BETWEEN 0 AND 720

    AND NULLIF(TRIM(t.treatmentString), '') IS NOT NULL

    AND REGEXP_CONTAINS(
      LOWER(t.treatmentString),
      r'mechanical ventilation|invasive ventilation|intubat|ventilator weaning'
    )

    AND NOT REGEXP_CONTAINS(
      LOWER(t.treatmentString),
      r'non.?invasive ventilation|\\bbipap\\b|bi-pap'
    )
)

SELECT
  treatment_item,

  COUNT(*) AS records,

  COUNT(DISTINCT mt.patientUnitStayID)
    AS patient_stays,

  COUNT(
    DISTINCT IF(
      COALESCE(da.invasive_airway, 0) = 1,
      mt.patientUnitStayID,
      NULL
    )
  ) AS patients_with_documented_invasive_airway,

  COUNT(
    DISTINCT IF(
      COALESCE(da.invasive_airway, 0) = 0,
      mt.patientUnitStayID,
      NULL
    )
  ) AS patients_without_documented_invasive_airway

FROM matched_treatments AS mt

LEFT JOIN direct_airway AS da
  ON da.patientUnitStayID = mt.patientUnitStayID

GROUP BY treatment_item

ORDER BY patient_stays DESC, treatment_item

LIMIT 150;
"""

result_04C = run_aggregate(
    "04C_respiratory_term_validation_audit.sql",
    SQL_04C_RESP_TERM_AUDIT,
    "04C_respiratory_term_validation_audit.csv"
)

## 05 Create Feature Matrix V1

In [ ]:
def run_ddl(label, sql):
    print("Running:", label)

    query_job = client.query(
        render_sql(sql),
        location=BQ_LOCATION
    )
    query_job.result()

    print("Completed:", label)

In [ ]:
SQL_05 = """
-- Final secure feature matrix.
-- The full table retains audit-only fields for subgroup and QC analyses.
-- The modelling view excludes direct identifiers, audit-only attributes
-- and the zero-variance high-flow oxygen variable.

CREATE OR REPLACE TABLE `{{TARGET_DATASET}}.feature_matrix_v1` AS

SELECT
  s.*,
  l.* EXCEPT (patientUnitStayID),
  v.* EXCEPT (patientUnitStayID),
  c.* EXCEPT (patientUnitStayID)

FROM `{{TARGET_DATASET}}.feature_static_v1` AS s

LEFT JOIN `{{TARGET_DATASET}}.feature_labs_v1` AS l
  ON l.patientUnitStayID = s.patientUnitStayID

LEFT JOIN `{{TARGET_DATASET}}.feature_vitals_v1` AS v
  ON v.patientUnitStayID = s.patientUnitStayID

LEFT JOIN `{{TARGET_DATASET}}.feature_context_v1` AS c
  ON c.patientUnitStayID = s.patientUnitStayID;


CREATE OR REPLACE VIEW
  `{{TARGET_DATASET}}.feature_matrix_model_view_v1` AS

SELECT * EXCEPT (
  patientUnitStayID,
  hospitalID,

  audit_ethnicity,
  audit_hospital_region,
  audit_hospital_bed_category,
  audit_hospital_teaching_status,
  audit_reference_offset,

  x_resp_high_flow_oxygen
)

FROM `{{TARGET_DATASET}}.feature_matrix_v1`;
"""



In [ ]:
from google.colab import auth
import google.auth
from google.cloud import bigquery

# Google hesabını yeniden yetkilendir
auth.authenticate_user()
PROJECT_ID = globals().get("PROJECT_ID") or os.environ.get("GOOGLE_CLOUD_PROJECT") or input("Enter your Google Cloud project ID: ").strip()
SOURCE_DATASET = "physionet-data.eicu_crd"
TARGET_DATASET = f"{PROJECT_ID}.{WORK_DATASET_NAME}"
BQ_LOCATION = "US"

# Kullanıcı yetkilendirmesini açık biçimde al
credentials, authenticated_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

# BigQuery istemcisini kullanıcı kimliğiyle yeniden oluştur
client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

# Bağlantı testi
test_result = client.query(
    "SELECT 1 AS connection_ok",
    location=BQ_LOCATION
).to_dataframe()

display(test_result)

print("BigQuery authentication renewed successfully.")

In [ ]:
if "SQL_05" not in globals():
    raise RuntimeError(
        "SQL_05 tanımlı değil. Önce uzun SQL_05 hücresini çalıştır."
    )

print("Running: 05_create_feature_matrix_v1.sql")

sql_05_ready = (
    SQL_05
    .replace("{{TARGET_DATASET}}", TARGET_DATASET)
    .replace("{{SOURCE_DATASET}}", SOURCE_DATASET)
)

query_job = client.query(
    sql_05_ready,
    location=BQ_LOCATION
)

query_job.result()

print("Completed: 05_create_feature_matrix_v1.sql")

In [ ]:
import os
from IPython.display import display

SQL_05A_AUDIT = f"""
WITH full_matrix AS (
  SELECT *
  FROM `{TARGET_DATASET}.feature_matrix_v1`
),

model_view AS (
  SELECT *
  FROM `{TARGET_DATASET}.feature_matrix_model_view_v1`
),

column_inventory AS (
  SELECT
    table_name,
    column_name
  FROM `{TARGET_DATASET}.INFORMATION_SCHEMA.COLUMNS`
  WHERE table_name IN (
    'feature_matrix_v1',
    'feature_matrix_model_view_v1'
  )
)

SELECT
  -- Full feature matrix integrity
  (SELECT COUNT(*) FROM full_matrix)
    AS full_matrix_rows,

  (
    SELECT COUNT(DISTINCT patientUnitStayID)
    FROM full_matrix
  ) AS distinct_unit_stays,

  (
    SELECT COUNT(DISTINCT id_row)
    FROM full_matrix
  ) AS distinct_row_keys,

  (
    SELECT COUNT(DISTINCT group_hospital)
    FROM full_matrix
  ) AS hospital_groups,

  (
    SELECT COUNTIF(label_stage23 = 1)
    FROM full_matrix
  ) AS events,

  (
    SELECT COUNTIF(label_stage23 = 0)
    FROM full_matrix
  ) AS nonevents,

  SAFE_DIVIDE(
    (SELECT COUNTIF(label_stage23 = 1) FROM full_matrix),
    (SELECT COUNT(*) FROM full_matrix)
  ) AS event_rate,

  (
    SELECT COUNT(*) - COUNT(DISTINCT patientUnitStayID)
    FROM full_matrix
  ) AS duplicate_stay_rows,

  (
    SELECT COUNTIF(id_row IS NULL)
    FROM full_matrix
  ) AS missing_row_keys,

  (
    SELECT COUNTIF(group_hospital IS NULL)
    FROM full_matrix
  ) AS missing_hospital_groups,

  (
    SELECT COUNTIF(label_stage23 IS NULL)
    FROM full_matrix
  ) AS missing_labels,

  -- Modelling-view integrity
  (SELECT COUNT(*) FROM model_view)
    AS model_view_rows,

  (
    SELECT COUNT(*)
    FROM column_inventory
    WHERE table_name = 'feature_matrix_v1'
  ) AS full_matrix_columns,

  (
    SELECT COUNT(*)
    FROM column_inventory
    WHERE table_name = 'feature_matrix_model_view_v1'
  ) AS model_view_columns,

  (
    SELECT COUNT(*)
    FROM column_inventory
    WHERE table_name = 'feature_matrix_model_view_v1'
      AND STARTS_WITH(column_name, 'x_')
  ) AS model_candidate_columns,

  (
    SELECT COUNT(*)
    FROM column_inventory
    WHERE table_name = 'feature_matrix_model_view_v1'
      AND STARTS_WITH(column_name, 'audit_')
  ) AS audit_columns_in_model_view,

  (
    SELECT COUNT(*)
    FROM column_inventory
    WHERE table_name = 'feature_matrix_model_view_v1'
      AND column_name IN (
        'patientUnitStayID',
        'hospitalID'
      )
  ) AS direct_identifier_columns_in_model_view,

  (
    SELECT COUNT(*)
    FROM column_inventory
    WHERE table_name = 'feature_matrix_model_view_v1'
      AND column_name = 'x_resp_high_flow_oxygen'
  ) AS zero_variance_high_flow_column_present,

  (
    SELECT COUNT(*)
    FROM column_inventory
    WHERE table_name = 'feature_matrix_model_view_v1'
      AND column_name IN (
        'id_row',
        'group_hospital',
        'label_stage23'
      )
  ) AS required_control_columns_present;
"""

print("Running: 05A_final_matrix_integrity_audit.sql")

result_05A = client.query(
    SQL_05A_AUDIT,
    location=BQ_LOCATION
).to_dataframe()

display(result_05A)

output_dir = (
    f"{OUTPUT_ROOT}/"
    "04_FEATURE_MATRIX_OUTPUTS"
)
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "05A_final_matrix_integrity_audit.csv"
)

result_05A.to_csv(output_path, index=False)

print("Saved:", output_path, "rows=", len(result_05A))

In [ ]:
import os
import re
import pandas as pd
from IPython.display import display

MODEL_VIEW = (
    f"{TARGET_DATASET}.feature_matrix_model_view_v1"
)

# ------------------------------------------------------------
# 1. Predictor sütunlarını BigQuery şemasından al
# ------------------------------------------------------------

schema_sql = f"""
SELECT
  column_name,
  data_type,
  ordinal_position
FROM `{TARGET_DATASET}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'feature_matrix_model_view_v1'
  AND STARTS_WITH(column_name, 'x_')
ORDER BY ordinal_position
"""

feature_schema = client.query(
    schema_sql,
    location=BQ_LOCATION
).to_dataframe()

print("Predictor columns found:", len(feature_schema))

if len(feature_schema) != 251:
    raise RuntimeError(
        f"Expected 251 predictor columns, found {len(feature_schema)}."
    )

# ------------------------------------------------------------
# 2. Her predictor için yalnızca toplulaştırılmış istatistik üret
# ------------------------------------------------------------

query_parts = []

for row in feature_schema.itertuples(index=False):
    column_name = row.column_name
    data_type = row.data_type

    # Şema kaynaklı sütun adlarını güvenli biçimde kullan
    safe_column = column_name.replace("`", "")

    query_parts.append(
        f"""
        SELECT
          '{safe_column}' AS feature,
          '{data_type}' AS data_type,

          COUNT(*) AS cohort_rows,

          COUNTIF(`{safe_column}` IS NULL)
            AS missing_count,

          COUNTIF(`{safe_column}` IS NOT NULL)
            AS nonmissing_count,

          SAFE_DIVIDE(
            COUNTIF(`{safe_column}` IS NULL),
            COUNT(*)
          ) AS missing_rate,

          COUNT(
            DISTINCT CAST(`{safe_column}` AS STRING)
          ) AS distinct_nonnull,

          MIN(
            SAFE_CAST(`{safe_column}` AS FLOAT64)
          ) AS numeric_min,

          MAX(
            SAFE_CAST(`{safe_column}` AS FLOAT64)
          ) AS numeric_max,

          COUNTIF(
            SAFE_CAST(`{safe_column}` AS FLOAT64) = 0
          ) AS numeric_zero_count,

          COUNTIF(
            SAFE_CAST(`{safe_column}` AS FLOAT64) = 1
          ) AS numeric_one_count,

          SAFE_DIVIDE(
            COUNTIF(
              SAFE_CAST(`{safe_column}` AS FLOAT64) = 1
            ),
            COUNT(*)
          ) AS numeric_one_prevalence

        FROM `{MODEL_VIEW}`
        """
    )

feature_audit_sql = "\nUNION ALL\n".join(query_parts)

print("Running: 05B_predictor_quality_audit.sql")

result_05B = client.query(
    feature_audit_sql,
    location=BQ_LOCATION
).to_dataframe()

# ------------------------------------------------------------
# 3. Kalite bayraklarını oluştur
# ------------------------------------------------------------

result_05B["all_missing"] = (
    result_05B["nonmissing_count"] == 0
)

result_05B["zero_variance"] = (
    result_05B["distinct_nonnull"] <= 1
)

result_05B["high_missingness_ge_70pct"] = (
    result_05B["missing_rate"] >= 0.70
)

result_05B["is_binary_01"] = (
    result_05B["numeric_min"].fillna(-999).ge(0)
    & result_05B["numeric_max"].fillna(999).le(1)
    & result_05B["distinct_nonnull"].le(2)
)

result_05B["rare_binary_lt_0_1pct"] = (
    result_05B["is_binary_01"]
    & (
        (result_05B["numeric_one_prevalence"] < 0.001)
        | (result_05B["numeric_one_prevalence"] > 0.999)
    )
)

leakage_pattern = re.compile(
    r"label|outcome|future|after_landmark|post_landmark|"
    r"stage23|discharge|mortality|death|length_of_stay|"
    r"(^|_)los($|_)|dialysis|(^|_)rrt($|_)",
    flags=re.IGNORECASE
)

result_05B["possible_name_leakage"] = (
    result_05B["feature"]
    .astype(str)
    .str.contains(leakage_pattern, regex=True)
)

# ------------------------------------------------------------
# 4. Özet
# ------------------------------------------------------------

summary_05B = pd.DataFrame(
    {
        "metric": [
            "predictor_columns",
            "all_missing_columns",
            "zero_variance_columns",
            "high_missingness_ge_70pct",
            "binary_01_columns",
            "rare_binary_lt_0_1pct",
            "possible_name_leakage_columns",
        ],
        "value": [
            len(result_05B),
            int(result_05B["all_missing"].sum()),
            int(result_05B["zero_variance"].sum()),
            int(result_05B["high_missingness_ge_70pct"].sum()),
            int(result_05B["is_binary_01"].sum()),
            int(result_05B["rare_binary_lt_0_1pct"].sum()),
            int(result_05B["possible_name_leakage"].sum()),
        ],
    }
)

flag_mask = (
    result_05B["all_missing"]
    | result_05B["zero_variance"]
    | result_05B["high_missingness_ge_70pct"]
    | result_05B["rare_binary_lt_0_1pct"]
    | result_05B["possible_name_leakage"]
)

flagged_05B = (
    result_05B.loc[flag_mask]
    .sort_values(
        by=[
            "possible_name_leakage",
            "zero_variance",
            "all_missing",
            "high_missingness_ge_70pct",
            "missing_rate",
        ],
        ascending=[False, False, False, False, False],
    )
    .reset_index(drop=True)
)

print("\n05B SUMMARY")
display(summary_05B)

print("\nFLAGGED PREDICTORS")
display(flagged_05B.head(100))

# ------------------------------------------------------------
# 5. Yalnızca aggregate sonuçları Drive'a kaydet
# ------------------------------------------------------------

output_dir = (
    f"{OUTPUT_ROOT}/"
    "04_FEATURE_MATRIX_OUTPUTS"
)
os.makedirs(output_dir, exist_ok=True)

result_05B.to_csv(
    os.path.join(
        output_dir,
        "05B_predictor_quality_audit.csv"
    ),
    index=False
)

summary_05B.to_csv(
    os.path.join(
        output_dir,
        "05B_predictor_quality_summary.csv"
    ),
    index=False
)

flagged_05B.to_csv(
    os.path.join(
        output_dir,
        "05B_flagged_predictors.csv"
    ),
    index=False
)

print(
    "\nSaved aggregate predictor audits to:",
    output_dir
)

In [ ]:
import os
import re
import hashlib
import pandas as pd
from IPython.display import display

OUTPUT_DIR = (
    f"{OUTPUT_ROOT}/"
    "04_FEATURE_MATRIX_OUTPUTS"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "05B_predictor_quality_audit.csv"
)

# ------------------------------------------------------------
# 1. Önceki audit sonucunu güvenli biçimde yükle
# ------------------------------------------------------------

if "result_05B" in globals():
    predictor_audit = result_05B.copy()
elif os.path.exists(AUDIT_PATH):
    predictor_audit = pd.read_csv(AUDIT_PATH)
else:
    raise FileNotFoundError(
        "05B_predictor_quality_audit.csv bulunamadı. "
        "Önce 05B hücresini çalıştır."
    )

# Şemayı BigQuery'den yeniden al
schema_sql = f"""
SELECT
  column_name,
  data_type,
  ordinal_position
FROM `{TARGET_DATASET}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'feature_matrix_model_view_v1'
  AND STARTS_WITH(column_name, 'x_')
ORDER BY ordinal_position
"""

feature_schema_locked = client.query(
    schema_sql,
    location=BQ_LOCATION
).to_dataframe()

if len(feature_schema_locked) != 251:
    raise RuntimeError(
        f"251 predictor bekleniyordu; "
        f"{len(feature_schema_locked)} bulundu."
    )

registry = feature_schema_locked.merge(
    predictor_audit,
    left_on="column_name",
    right_on="feature",
    how="left",
    validate="one_to_one"
)

if registry["feature"].isna().any():
    missing_audits = registry.loc[
        registry["feature"].isna(),
        "column_name"
    ].tolist()

    raise RuntimeError(
        "Audit sonucu bulunmayan predictorlar var: "
        + ", ".join(missing_audits[:20])
    )

# ------------------------------------------------------------
# 2. Audit bayraklarını yeniden ve tutarlı biçimde hesapla
# ------------------------------------------------------------

registry["all_missing"] = (
    registry["nonmissing_count"] == 0
)

registry["zero_variance"] = (
    registry["distinct_nonnull"] <= 1
)

registry["high_missingness_ge_70pct"] = (
    registry["missing_rate"] >= 0.70
)

registry["is_binary_01"] = (
    registry["numeric_min"].fillna(-999).ge(0)
    & registry["numeric_max"].fillna(999).le(1)
    & registry["distinct_nonnull"].le(2)
)

registry["rare_binary_lt_0_1pct"] = (
    registry["is_binary_01"]
    & (
        (registry["numeric_one_prevalence"] < 0.001)
        | (registry["numeric_one_prevalence"] > 0.999)
    )
)

# Parantez uyarısı oluşturmayan non-capturing regex
leakage_pattern = re.compile(
    r"label|outcome|future|after_landmark|post_landmark|"
    r"stage23|discharge|mortality|death|length_of_stay|"
    r"(?:^|_)los(?:$|_)|dialysis|(?:^|_)rrt(?:$|_)",
    flags=re.IGNORECASE
)

registry["possible_name_leakage"] = (
    registry["column_name"]
    .astype(str)
    .str.contains(leakage_pattern, regex=True)
)

# ------------------------------------------------------------
# 3. Protokoldeki core değişken ailelerini tanımla
# ------------------------------------------------------------

core_exact = {
    "x_age_years",
    "x_sex",
    "x_bmi",
    "x_unit_type",
    "x_unit_admit_source",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
}

core_lab_groups = [
    "creatinine",
    "bun",
    "bicarbonate",
    "potassium",
    "sodium",
    "chloride",
    "glucose_serum",
    "calcium",
    "magnesium",
    "phosphate",
    "hgb",
    "hct",
    "platelets",
    "wbc",
    "albumin",
    "bilirubin_total",
    "inr",
    "lactate",
    "anion_gap",
]

core_vital_groups = [
    "heart_rate",
    "sao2",
    "respiratory_rate",
    "noninvasive_systolic_bp",
    "noninvasive_diastolic_bp",
    "noninvasive_mean_bp",
]

core_prefixes = tuple(
    [f"x_lab_{name}_" for name in core_lab_groups]
    + [f"x_vital_{name}_" for name in core_vital_groups]
)

def protocol_tier(feature_name):
    if feature_name in core_exact:
        return "core_candidate"

    if feature_name.startswith(core_prefixes):
        return "core_candidate"

    return "extended_candidate"

registry["protocol_tier"] = (
    registry["column_name"]
    .map(protocol_tier)
)

# ------------------------------------------------------------
# 4. Nihai dahil etme / dışlama kararları
# ------------------------------------------------------------

def exclusion_reason(row):
    if row["all_missing"]:
        return "all_missing"

    if row["zero_variance"]:
        return "zero_variance"

    if row["possible_name_leakage"]:
        return "possible_name_leakage"

    if row["rare_binary_lt_0_1pct"]:
        return "rare_binary_prevalence_lt_0_1pct"

    return ""

registry["exclusion_reason"] = registry.apply(
    exclusion_reason,
    axis=1
)

def assign_final_tier(row):
    if row["exclusion_reason"] != "":
        return "excluded"

    # Core adayları yüksek eksiklik nedeniyle extended'a taşınır
    if (
        row["protocol_tier"] == "core_candidate"
        and row["missing_rate"] < 0.70
    ):
        return "core"

    return "extended"

registry["final_tier"] = registry.apply(
    assign_final_tier,
    axis=1
)

registry["moved_from_core_due_high_missingness"] = (
    (registry["protocol_tier"] == "core_candidate")
    & (registry["final_tier"] == "extended")
    & (registry["high_missingness_ge_70pct"])
)

registry["included_in_extended_model"] = (
    registry["final_tier"].isin(["core", "extended"])
)

registry["included_in_core_model"] = (
    registry["final_tier"] == "core"
)

registry = registry.sort_values(
    "ordinal_position"
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. Kilitli sütun listelerini oluştur
# ------------------------------------------------------------

core_columns = registry.loc[
    registry["included_in_core_model"],
    "column_name"
].tolist()

extended_columns = registry.loc[
    registry["included_in_extended_model"],
    "column_name"
].tolist()

excluded_columns = registry.loc[
    registry["final_tier"] == "excluded",
    "column_name"
].tolist()

if "x_tx_bicarbonate_infusion" not in excluded_columns:
    raise RuntimeError(
        "x_tx_bicarbonate_infusion beklenen şekilde "
        "excluded olarak işaretlenmedi."
    )

# Kontrol alanları predictor değildir
control_columns = [
    "id_row",
    "group_hospital",
    "label_stage23",
]

# ------------------------------------------------------------
# 6. BigQuery core ve extended model görünümlerini oluştur
# ------------------------------------------------------------

def quoted_columns(columns):
    return ",\n  ".join(
        f"`{column}`"
        for column in columns
    )

core_view_sql = f"""
CREATE OR REPLACE VIEW
  `{TARGET_DATASET}.feature_matrix_core_view_v1` AS

SELECT
  {quoted_columns(control_columns + core_columns)}

FROM `{TARGET_DATASET}.feature_matrix_model_view_v1`;
"""

extended_view_sql = f"""
CREATE OR REPLACE VIEW
  `{TARGET_DATASET}.feature_matrix_extended_view_v1` AS

SELECT
  {quoted_columns(control_columns + extended_columns)}

FROM `{TARGET_DATASET}.feature_matrix_model_view_v1`;
"""

print("Creating: feature_matrix_core_view_v1")
client.query(
    core_view_sql,
    location=BQ_LOCATION
).result()

print("Creating: feature_matrix_extended_view_v1")
client.query(
    extended_view_sql,
    location=BQ_LOCATION
).result()

print("Locked model views created.")

# ------------------------------------------------------------
# 7. Registry ve SHA-256 kilidi
# ------------------------------------------------------------

registry_output_columns = [
    "ordinal_position",
    "column_name",
    "data_type",
    "missing_count",
    "nonmissing_count",
    "missing_rate",
    "distinct_nonnull",
    "numeric_min",
    "numeric_max",
    "numeric_one_prevalence",
    "protocol_tier",
    "final_tier",
    "included_in_core_model",
    "included_in_extended_model",
    "moved_from_core_due_high_missingness",
    "exclusion_reason",
]

registry_path = os.path.join(
    OUTPUT_DIR,
    "05C_locked_predictor_registry_v1.csv"
)

registry[
    registry_output_columns
].to_csv(
    registry_path,
    index=False
)

with open(registry_path, "rb") as file_handle:
    registry_sha256 = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

sha_path = os.path.join(
    OUTPUT_DIR,
    "05C_locked_predictor_registry_v1_SHA256.txt"
)

with open(sha_path, "w", encoding="utf-8") as file_handle:
    file_handle.write(
        registry_sha256 + "\n"
    )

# ------------------------------------------------------------
# 8. Özet ve doğrulama
# ------------------------------------------------------------

summary_05C = pd.DataFrame(
    {
        "metric": [
            "original_predictor_columns",
            "core_locked_predictors",
            "extended_total_predictors",
            "extended_only_predictors",
            "excluded_predictors",
            "core_candidates_moved_due_missingness",
        ],
        "value": [
            len(registry),
            len(core_columns),
            len(extended_columns),
            len(extended_columns) - len(core_columns),
            len(excluded_columns),
            int(
                registry[
                    "moved_from_core_due_high_missingness"
                ].sum()
            ),
        ],
    }
)

excluded_05C = registry.loc[
    registry["final_tier"] == "excluded",
    [
        "column_name",
        "missing_rate",
        "numeric_one_prevalence",
        "exclusion_reason",
    ]
].reset_index(drop=True)

moved_05C = registry.loc[
    registry["moved_from_core_due_high_missingness"],
    [
        "column_name",
        "missing_rate",
        "protocol_tier",
        "final_tier",
    ]
].sort_values(
    "missing_rate",
    ascending=False
).reset_index(drop=True)

print("\n05C LOCKED PREDICTOR SUMMARY")
display(summary_05C)

print("\nEXCLUDED PREDICTORS")
display(excluded_05C)

print("\nMOVED FROM CORE TO EXTENDED")
display(moved_05C)

print("\nRegistry SHA-256:")
print(registry_sha256)

print("\nSaved:")
print(registry_path)
print(sha_path)

In [ ]:
import os
import hashlib
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# 05C kayıt aşaması düzeltmesi
# BigQuery core ve extended view'ları zaten oluşturuldu.
# Bu hücre yalnızca registry sütununu düzeltir,
# CSV/SHA dosyalarını ve özetleri üretir.
# ------------------------------------------------------------

if "registry" not in globals():
    raise RuntimeError(
        "registry değişkeni bulunamadı. "
        "Colab oturumu yeniden başlamış olabilir. "
        "Bu durumda 05B ve düzeltilmiş 05C tekrar çalıştırılmalıdır."
    )

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = (
        f"{OUTPUT_ROOT}/"
        "04_FEATURE_MATRIX_OUTPUTS"
    )

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Merge sonrası data_type sütunu data_type_x / data_type_y olmuş olabilir.
if "data_type" not in registry.columns:

    if "data_type_x" in registry.columns:
        registry["data_type"] = registry["data_type_x"]

    elif "data_type_y" in registry.columns:
        registry["data_type"] = registry["data_type_y"]

    else:
        raise KeyError(
            "Registry içinde data_type, data_type_x veya "
            "data_type_y sütunu bulunamadı."
        )

# Kilitli predictor listelerini registry'den yeniden oluştur.
core_columns = registry.loc[
    registry["included_in_core_model"] == True,
    "column_name"
].tolist()

extended_columns = registry.loc[
    registry["included_in_extended_model"] == True,
    "column_name"
].tolist()

excluded_columns = registry.loc[
    registry["final_tier"] == "excluded",
    "column_name"
].tolist()

# Kaydedilecek registry sütunları
registry_output_columns = [
    "ordinal_position",
    "column_name",
    "data_type",
    "missing_count",
    "nonmissing_count",
    "missing_rate",
    "distinct_nonnull",
    "numeric_min",
    "numeric_max",
    "numeric_one_prevalence",
    "protocol_tier",
    "final_tier",
    "included_in_core_model",
    "included_in_extended_model",
    "moved_from_core_due_high_missingness",
    "exclusion_reason",
]

missing_registry_columns = [
    column
    for column in registry_output_columns
    if column not in registry.columns
]

if missing_registry_columns:
    raise KeyError(
        "Registry içinde eksik sütunlar var: "
        + ", ".join(missing_registry_columns)
    )

# Registry CSV
registry_path = os.path.join(
    OUTPUT_DIR,
    "05C_locked_predictor_registry_v1.csv"
)

registry[
    registry_output_columns
].to_csv(
    registry_path,
    index=False
)

# SHA-256
with open(registry_path, "rb") as file_handle:
    registry_sha256 = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

sha_path = os.path.join(
    OUTPUT_DIR,
    "05C_locked_predictor_registry_v1_SHA256.txt"
)

with open(sha_path, "w", encoding="utf-8") as file_handle:
    file_handle.write(registry_sha256 + "\n")

# Özet
summary_05C = pd.DataFrame(
    {
        "metric": [
            "original_predictor_columns",
            "core_locked_predictors",
            "extended_total_predictors",
            "extended_only_predictors",
            "excluded_predictors",
            "core_candidates_moved_due_missingness",
        ],
        "value": [
            len(registry),
            len(core_columns),
            len(extended_columns),
            len(extended_columns) - len(core_columns),
            len(excluded_columns),
            int(
                registry[
                    "moved_from_core_due_high_missingness"
                ].sum()
            ),
        ],
    }
)

excluded_05C = registry.loc[
    registry["final_tier"] == "excluded",
    [
        "column_name",
        "missing_rate",
        "numeric_one_prevalence",
        "exclusion_reason",
    ]
].reset_index(drop=True)

moved_05C = registry.loc[
    registry["moved_from_core_due_high_missingness"] == True,
    [
        "column_name",
        "missing_rate",
        "protocol_tier",
        "final_tier",
    ]
].sort_values(
    "missing_rate",
    ascending=False
).reset_index(drop=True)

# Ek bütünlük doğrulamaları
if len(registry) != 251:
    raise RuntimeError(
        f"251 predictor bekleniyordu; {len(registry)} bulundu."
    )

if "x_tx_bicarbonate_infusion" not in excluded_columns:
    raise RuntimeError(
        "x_tx_bicarbonate_infusion excluded listesinde bulunamadı."
    )

print("05C LOCKED PREDICTOR SUMMARY")
display(summary_05C)

print("\nEXCLUDED PREDICTORS")
display(excluded_05C)

print("\nMOVED FROM CORE TO EXTENDED")
display(moved_05C)

print("\nRegistry SHA-256:")
print(registry_sha256)

print("\nSaved:")
print(registry_path)
print(sha_path)

In [ ]:
import os
import pandas as pd
from IPython.display import display

TARGET_DATASET = globals().get(
    "TARGET_DATASET",
    f"{PROJECT_ID}.aki_jcmc_v2"
)

BQ_LOCATION = globals().get(
    "BQ_LOCATION",
    "US"
)

OUTPUT_DIR = (
    f"{OUTPUT_ROOT}/"
    "04_FEATURE_MATRIX_OUTPUTS"
)

REGISTRY_PATH = os.path.join(
    OUTPUT_DIR,
    "05C_locked_predictor_registry_v1.csv"
)

if not os.path.exists(REGISTRY_PATH):
    raise FileNotFoundError(
        "05C locked predictor registry bulunamadı: "
        + REGISTRY_PATH
    )

registry_05D = pd.read_csv(REGISTRY_PATH)

required_registry_columns = {
    "column_name",
    "final_tier",
    "included_in_core_model",
    "included_in_extended_model",
    "exclusion_reason",
}

missing_registry_columns = (
    required_registry_columns
    - set(registry_05D.columns)
)

if missing_registry_columns:
    raise RuntimeError(
        "Registry içinde eksik sütunlar var: "
        + ", ".join(sorted(missing_registry_columns))
    )

control_columns = {
    "id_row",
    "group_hospital",
    "label_stage23",
}

expected_core = set(
    registry_05D.loc[
        registry_05D["included_in_core_model"] == True,
        "column_name"
    ]
)

expected_extended = set(
    registry_05D.loc[
        registry_05D[
            "included_in_extended_model"
        ] == True,
        "column_name"
    ]
)

expected_excluded = set(
    registry_05D.loc[
        registry_05D["final_tier"] == "excluded",
        "column_name"
    ]
)

# ------------------------------------------------------------
# 1. BigQuery view şemalarını oku
# ------------------------------------------------------------

schema_sql = f"""
SELECT
  table_name,
  column_name,
  ordinal_position
FROM `{TARGET_DATASET}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name IN (
  'feature_matrix_core_view_v1',
  'feature_matrix_extended_view_v1'
)
ORDER BY table_name, ordinal_position
"""

schema_05D = client.query(
    schema_sql,
    location=BQ_LOCATION
).to_dataframe()

core_view_columns = set(
    schema_05D.loc[
        schema_05D["table_name"]
        == "feature_matrix_core_view_v1",
        "column_name"
    ]
)

extended_view_columns = set(
    schema_05D.loc[
        schema_05D["table_name"]
        == "feature_matrix_extended_view_v1",
        "column_name"
    ]
)

actual_core = (
    core_view_columns - control_columns
)

actual_extended = (
    extended_view_columns - control_columns
)

# ------------------------------------------------------------
# 2. Satır ve etiket bütünlüğü
# ------------------------------------------------------------

row_sql = f"""
SELECT
  'core' AS model_set,
  COUNT(*) AS table_rows,
  COUNT(DISTINCT id_row) AS distinct_row_keys,
  COUNT(DISTINCT group_hospital) AS hospital_groups,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(label_stage23 IS NULL) AS missing_labels,
  COUNT(*) - COUNT(DISTINCT id_row) AS duplicate_row_keys
FROM `{TARGET_DATASET}.feature_matrix_core_view_v1`

UNION ALL

SELECT
  'extended' AS model_set,
  COUNT(*) AS table_rows,
  COUNT(DISTINCT id_row) AS distinct_row_keys,
  COUNT(DISTINCT group_hospital) AS hospital_groups,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(label_stage23 IS NULL) AS missing_labels,
  COUNT(*) - COUNT(DISTINCT id_row) AS duplicate_row_keys
FROM `{TARGET_DATASET}.feature_matrix_extended_view_v1`

ORDER BY model_set;
"""

row_audit_05D = client.query(
    row_sql,
    location=BQ_LOCATION
).to_dataframe()

# ------------------------------------------------------------
# 3. Şema uyuşmazlıklarını hesapla
# ------------------------------------------------------------

core_missing = sorted(
    expected_core - actual_core
)

core_extra = sorted(
    actual_core - expected_core
)

extended_missing = sorted(
    expected_extended - actual_extended
)

extended_extra = sorted(
    actual_extended - expected_extended
)

excluded_present_core = sorted(
    expected_excluded & actual_core
)

excluded_present_extended = sorted(
    expected_excluded & actual_extended
)

core_not_in_extended = sorted(
    actual_core - actual_extended
)

missing_core_controls = sorted(
    control_columns - core_view_columns
)

missing_extended_controls = sorted(
    control_columns - extended_view_columns
)

mismatch_05D = pd.DataFrame(
    {
        "check": [
            "core_missing_predictors",
            "core_extra_predictors",
            "extended_missing_predictors",
            "extended_extra_predictors",
            "excluded_present_in_core",
            "excluded_present_in_extended",
            "core_predictors_not_in_extended",
            "missing_core_control_columns",
            "missing_extended_control_columns",
        ],
        "count": [
            len(core_missing),
            len(core_extra),
            len(extended_missing),
            len(extended_extra),
            len(excluded_present_core),
            len(excluded_present_extended),
            len(core_not_in_extended),
            len(missing_core_controls),
            len(missing_extended_controls),
        ],
        "items": [
            "; ".join(core_missing),
            "; ".join(core_extra),
            "; ".join(extended_missing),
            "; ".join(extended_extra),
            "; ".join(excluded_present_core),
            "; ".join(excluded_present_extended),
            "; ".join(core_not_in_extended),
            "; ".join(missing_core_controls),
            "; ".join(missing_extended_controls),
        ],
    }
)

summary_05D = pd.DataFrame(
    {
        "metric": [
            "registry_predictors",
            "expected_core_predictors",
            "actual_core_predictors",
            "expected_extended_predictors",
            "actual_extended_predictors",
            "excluded_predictors",
            "core_total_columns_with_controls",
            "extended_total_columns_with_controls",
            "schema_mismatch_total",
        ],
        "value": [
            len(registry_05D),
            len(expected_core),
            len(actual_core),
            len(expected_extended),
            len(actual_extended),
            len(expected_excluded),
            len(core_view_columns),
            len(extended_view_columns),
            int(mismatch_05D["count"].sum()),
        ],
    }
)

# ------------------------------------------------------------
# 4. Kesin geçiş koşulları
# ------------------------------------------------------------

expected_row_values = {
    "table_rows": 58491,
    "distinct_row_keys": 58491,
    "hospital_groups": 198,
    "events": 3032,
    "nonevents": 55459,
    "missing_labels": 0,
    "duplicate_row_keys": 0,
}

for _, row in row_audit_05D.iterrows():
    for field, expected_value in expected_row_values.items():
        actual_value = int(row[field])

        if actual_value != expected_value:
            raise RuntimeError(
                f"{row['model_set']} görünümünde "
                f"{field}={actual_value}; "
                f"beklenen={expected_value}"
            )

if int(mismatch_05D["count"].sum()) != 0:
    raise RuntimeError(
        "Core/extended view şemalarında registry ile "
        "uyuşmazlık bulundu. Mismatch tablosunu incele."
    )

if len(actual_core) != 159:
    raise RuntimeError(
        f"Core predictor sayısı 159 yerine "
        f"{len(actual_core)}."
    )

if len(actual_extended) != 250:
    raise RuntimeError(
        f"Extended predictor sayısı 250 yerine "
        f"{len(actual_extended)}."
    )

# ------------------------------------------------------------
# 5. Görüntüle ve toplulaştırılmış dosyaları kaydet
# ------------------------------------------------------------

print("05D LOCKED VIEW SUMMARY")
display(summary_05D)

print("\n05D ROW INTEGRITY")
display(row_audit_05D)

print("\n05D SCHEMA MISMATCH AUDIT")
display(mismatch_05D)

summary_05D.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "05D_locked_view_summary.csv"
    ),
    index=False
)

row_audit_05D.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "05D_locked_view_row_integrity.csv"
    ),
    index=False
)

mismatch_05D.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "05D_locked_view_schema_mismatch.csv"
    ),
    index=False
)

print("\n05D PASS: Core and extended views match the locked registry.")
print("Saved aggregate audit files to:", OUTPUT_DIR)

## 06 Feature matrix integrity audit

In [ ]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

TARGET_DATASET = globals().get(
    "TARGET_DATASET",
    f"{PROJECT_ID}.aki_jcmc_v2"
)

BQ_LOCATION = globals().get(
    "BQ_LOCATION",
    "US"
)

OUTPUT_DIR = globals().get(
    "OUTPUT_DIR",
    f"{OUTPUT_ROOT}/"
    "04_FEATURE_MATRIX_OUTPUTS"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Yalnızca hastane düzeyinde toplulaştırılmış veri
# ------------------------------------------------------------

SQL_06A = f"""
SELECT
  group_hospital,

  COUNT(*) AS patients,

  COUNTIF(label_stage23 = 1) AS events,

  COUNTIF(label_stage23 = 0) AS nonevents,

  SAFE_DIVIDE(
    COUNTIF(label_stage23 = 1),
    COUNT(*)
  ) AS event_rate

FROM `{TARGET_DATASET}.feature_matrix_core_view_v1`

GROUP BY group_hospital

ORDER BY events DESC, patients DESC, group_hospital;
"""

print("Running: 06A_hospital_validation_feasibility.sql")

hospital_profile_06A = client.query(
    SQL_06A,
    location=BQ_LOCATION
).to_dataframe()

print(
    "Hospital aggregate rows:",
    len(hospital_profile_06A)
)

# ------------------------------------------------------------
# 2. Bütünlük kontrolleri
# ------------------------------------------------------------

expected_values = {
    "hospitals": 198,
    "patients": 58491,
    "events": 3032,
    "nonevents": 55459,
}

actual_values = {
    "hospitals": len(hospital_profile_06A),
    "patients": int(hospital_profile_06A["patients"].sum()),
    "events": int(hospital_profile_06A["events"].sum()),
    "nonevents": int(hospital_profile_06A["nonevents"].sum()),
}

for metric, expected in expected_values.items():
    actual = actual_values[metric]

    if actual != expected:
        raise RuntimeError(
            f"{metric}: bulunan={actual}, beklenen={expected}"
        )

if hospital_profile_06A["group_hospital"].duplicated().any():
    raise RuntimeError(
        "Hastane aggregate tablosunda yinelenen grup bulundu."
    )

# ------------------------------------------------------------
# 3. Hastane olay dağılımı
# ------------------------------------------------------------

event_bins = [-1, 0, 4, 9, 19, 49, np.inf]

event_labels = [
    "0 events",
    "1-4 events",
    "5-9 events",
    "10-19 events",
    "20-49 events",
    "50+ events",
]

hospital_profile_06A["event_count_band"] = pd.cut(
    hospital_profile_06A["events"],
    bins=event_bins,
    labels=event_labels
)

event_distribution_06A = (
    hospital_profile_06A[
        "event_count_band"
    ]
    .value_counts(sort=False)
    .rename_axis("event_count_band")
    .reset_index(name="hospitals")
)

event_distribution_06A["hospital_percentage"] = (
    event_distribution_06A["hospitals"]
    / len(hospital_profile_06A)
)

# ------------------------------------------------------------
# 4. Fizibilite özeti
# ------------------------------------------------------------

summary_06A = pd.DataFrame(
    {
        "metric": [
            "total_hospitals",
            "total_patients",
            "total_events",
            "total_nonevents",
            "overall_event_rate",
            "hospitals_with_zero_events",
            "hospitals_with_at_least_1_event",
            "hospitals_with_at_least_5_events",
            "hospitals_with_at_least_10_events",
            "hospitals_with_at_least_20_events",
            "hospitals_with_at_least_50_events",
            "minimum_patients_per_hospital",
            "median_patients_per_hospital",
            "maximum_patients_per_hospital",
            "minimum_events_per_hospital",
            "median_events_per_hospital",
            "maximum_events_per_hospital",
        ],
        "value": [
            len(hospital_profile_06A),
            int(hospital_profile_06A["patients"].sum()),
            int(hospital_profile_06A["events"].sum()),
            int(hospital_profile_06A["nonevents"].sum()),
            (
                hospital_profile_06A["events"].sum()
                / hospital_profile_06A["patients"].sum()
            ),
            int(
                (hospital_profile_06A["events"] == 0).sum()
            ),
            int(
                (hospital_profile_06A["events"] >= 1).sum()
            ),
            int(
                (hospital_profile_06A["events"] >= 5).sum()
            ),
            int(
                (hospital_profile_06A["events"] >= 10).sum()
            ),
            int(
                (hospital_profile_06A["events"] >= 20).sum()
            ),
            int(
                (hospital_profile_06A["events"] >= 50).sum()
            ),
            int(hospital_profile_06A["patients"].min()),
            float(hospital_profile_06A["patients"].median()),
            int(hospital_profile_06A["patients"].max()),
            int(hospital_profile_06A["events"].min()),
            float(hospital_profile_06A["events"].median()),
            int(hospital_profile_06A["events"].max()),
        ],
    }
)

# ------------------------------------------------------------
# 5. Görüntüle
# ------------------------------------------------------------

print("\n06A HOSPITAL VALIDATION SUMMARY")
display(summary_06A)

print("\n06A HOSPITAL EVENT DISTRIBUTION")
display(event_distribution_06A)

print("\nLARGEST EVENT-CONTRIBUTING HOSPITALS")
display(
    hospital_profile_06A[
        [
            "patients",
            "events",
            "nonevents",
            "event_rate",
        ]
    ].head(20)
)

# ------------------------------------------------------------
# 6. Yalnızca hastane düzeyindeki aggregate sonuçları kaydet
# ------------------------------------------------------------

hospital_profile_path = os.path.join(
    OUTPUT_DIR,
    "06A_hospital_profile_aggregate.csv"
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "06A_hospital_validation_summary.csv"
)

distribution_path = os.path.join(
    OUTPUT_DIR,
    "06A_hospital_event_distribution.csv"
)

hospital_profile_06A.to_csv(
    hospital_profile_path,
    index=False
)

summary_06A.to_csv(
    summary_path,
    index=False
)

event_distribution_06A.to_csv(
    distribution_path,
    index=False
)

print("\n06A PASS: Hospital-level validation feasibility audited.")
print("Saved aggregate files to:", OUTPUT_DIR)

In [ ]:
import os
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display
from google.cloud import bigquery

# ============================================================
# 06B — Deterministic balanced hospital outer-fold assignment
# ============================================================

TARGET_DATASET = globals().get(
    "TARGET_DATASET",
    f"{PROJECT_ID}.aki_jcmc_v2"
)

BQ_LOCATION = globals().get(
    "BQ_LOCATION",
    "US"
)

OUTPUT_DIR = globals().get(
    "OUTPUT_DIR",
    f"{OUTPUT_ROOT}/"
    "04_FEATURE_MATRIX_OUTPUTS"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

PROFILE_PATH = os.path.join(
    OUTPUT_DIR,
    "06A_hospital_profile_aggregate.csv"
)

# ------------------------------------------------------------
# 1. Hastane aggregate profilini al
# ------------------------------------------------------------

if "hospital_profile_06A" in globals():
    hospital_profile = hospital_profile_06A.copy()

elif os.path.exists(PROFILE_PATH):
    hospital_profile = pd.read_csv(
        PROFILE_PATH,
        dtype={"group_hospital": str}
    )

else:
    raise FileNotFoundError(
        "06A_hospital_profile_aggregate.csv bulunamadı. "
        "Önce 06A hücresini çalıştır."
    )

required_columns = {
    "group_hospital",
    "patients",
    "events",
    "nonevents",
    "event_rate",
}

missing_columns = (
    required_columns - set(hospital_profile.columns)
)

if missing_columns:
    raise RuntimeError(
        "Hastane profilinde eksik sütunlar var: "
        + ", ".join(sorted(missing_columns))
    )

hospital_profile = hospital_profile[
    [
        "group_hospital",
        "patients",
        "events",
        "nonevents",
        "event_rate",
    ]
].copy()

hospital_profile["group_hospital"] = (
    hospital_profile["group_hospital"].astype(str)
)

for column in ["patients", "events", "nonevents"]:
    hospital_profile[column] = (
        hospital_profile[column].astype(int)
    )

hospital_profile["event_rate"] = (
    hospital_profile["event_rate"].astype(float)
)

if len(hospital_profile) != 198:
    raise RuntimeError(
        f"198 hastane bekleniyordu; "
        f"{len(hospital_profile)} bulundu."
    )

if hospital_profile["group_hospital"].duplicated().any():
    raise RuntimeError(
        "Hastane profilinde yinelenen group_hospital var."
    )

if int(hospital_profile["patients"].sum()) != 58491:
    raise RuntimeError("Toplam hasta sayısı 58.491 değil.")

if int(hospital_profile["events"].sum()) != 3032:
    raise RuntimeError("Toplam olay sayısı 3.032 değil.")

# ------------------------------------------------------------
# 2. Beş dış kat için hedefler
# ------------------------------------------------------------

N_FOLDS = 5
ASSIGNMENT_SEED = 20260721
N_TRIALS = 5000

total_patients = int(
    hospital_profile["patients"].sum()
)

total_events = int(
    hospital_profile["events"].sum()
)

total_hospitals = len(hospital_profile)

total_zero_event_hospitals = int(
    (hospital_profile["events"] == 0).sum()
)

target_patients = total_patients / N_FOLDS
target_events = total_events / N_FOLDS
target_hospitals = total_hospitals / N_FOLDS
target_zero_hospitals = (
    total_zero_event_hospitals / N_FOLDS
)

maximum_hospitals_per_fold = int(
    np.ceil(target_hospitals)
)

overall_event_rate = (
    total_events / total_patients
)

# ------------------------------------------------------------
# 3. Denge skor fonksiyonu
# ------------------------------------------------------------

def fold_balance_score(
    patient_totals,
    event_totals,
    hospital_totals,
    zero_event_hospital_totals
):
    patient_component = np.sum(
        (
            (patient_totals - target_patients)
            / target_patients
        ) ** 2
    )

    event_component = np.sum(
        (
            (event_totals - target_events)
            / target_events
        ) ** 2
    )

    hospital_component = np.sum(
        (
            (hospital_totals - target_hospitals)
            / target_hospitals
        ) ** 2
    )

    zero_event_component = np.sum(
        (
            (
                zero_event_hospital_totals
                - target_zero_hospitals
            )
            / max(target_zero_hospitals, 1)
        ) ** 2
    )

    fold_rates = np.divide(
        event_totals,
        patient_totals,
        out=np.zeros_like(
            event_totals,
            dtype=float
        ),
        where=patient_totals > 0
    )

    rate_component = np.sum(
        (
            (fold_rates - overall_event_rate)
            / overall_event_rate
        ) ** 2
    )

    return (
        2.0 * patient_component
        + 5.0 * event_component
        + 0.30 * hospital_component
        + 0.40 * zero_event_component
        + 1.50 * rate_component
    )

# ------------------------------------------------------------
# 4. Çoklu deterministik greedy arama
# ------------------------------------------------------------

rng_master = np.random.default_rng(
    ASSIGNMENT_SEED
)

best_score = np.inf
best_assignment = None

base_profile = hospital_profile.copy()

base_profile["_priority_base"] = (
    0.55
    * (
        base_profile["events"]
        / max(target_events, 1)
    )
    + 0.45
    * (
        base_profile["patients"]
        / max(target_patients, 1)
    )
)

for trial in range(N_TRIALS):

    trial_seed = int(
        rng_master.integers(
            0,
            np.iinfo(np.int32).max
        )
    )

    rng = np.random.default_rng(trial_seed)

    trial_profile = base_profile.copy()

    trial_profile["_jitter"] = rng.uniform(
        0.0,
        0.10,
        size=len(trial_profile)
    )

    trial_profile["_priority"] = (
        trial_profile["_priority_base"]
        + trial_profile["_jitter"]
    )

    trial_profile = trial_profile.sort_values(
        ["_priority", "events", "patients"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    fold_patients = np.zeros(
        N_FOLDS,
        dtype=float
    )

    fold_events = np.zeros(
        N_FOLDS,
        dtype=float
    )

    fold_hospitals = np.zeros(
        N_FOLDS,
        dtype=int
    )

    fold_zero_event_hospitals = np.zeros(
        N_FOLDS,
        dtype=int
    )

    assigned_folds = []

    for row in trial_profile.itertuples(
        index=False
    ):
        candidate_scores = []

        candidate_order = rng.permutation(
            N_FOLDS
        )

        for fold_index in candidate_order:

            if (
                fold_hospitals[fold_index]
                >= maximum_hospitals_per_fold
            ):
                continue

            candidate_patients = (
                fold_patients.copy()
            )

            candidate_events = (
                fold_events.copy()
            )

            candidate_hospitals = (
                fold_hospitals.copy()
            )

            candidate_zero_hospitals = (
                fold_zero_event_hospitals.copy()
            )

            candidate_patients[fold_index] += (
                row.patients
            )

            candidate_events[fold_index] += (
                row.events
            )

            candidate_hospitals[fold_index] += 1

            if row.events == 0:
                candidate_zero_hospitals[
                    fold_index
                ] += 1

            score = fold_balance_score(
                candidate_patients,
                candidate_events,
                candidate_hospitals,
                candidate_zero_hospitals
            )

            candidate_scores.append(
                (score, fold_index)
            )

        if not candidate_scores:
            raise RuntimeError(
                "Uygun dış kat bulunamadı."
            )

        candidate_scores.sort(
            key=lambda item: item[0]
        )

        selected_fold = (
            candidate_scores[0][1]
        )

        assigned_folds.append(
            selected_fold + 1
        )

        fold_patients[selected_fold] += (
            row.patients
        )

        fold_events[selected_fold] += (
            row.events
        )

        fold_hospitals[selected_fold] += 1

        if row.events == 0:
            fold_zero_event_hospitals[
                selected_fold
            ] += 1

    final_score = fold_balance_score(
        fold_patients,
        fold_events,
        fold_hospitals,
        fold_zero_event_hospitals
    )

    if final_score < best_score:
        best_score = final_score

        best_assignment = (
            trial_profile[
                [
                    "group_hospital",
                    "patients",
                    "events",
                    "nonevents",
                    "event_rate",
                ]
            ]
            .assign(
                outer_fold=assigned_folds
            )
            .copy()
        )

if best_assignment is None:
    raise RuntimeError(
        "Geçerli dış kat ataması üretilemedi."
    )

# ------------------------------------------------------------
# 5. Kilitli atama ve fold özeti
# ------------------------------------------------------------

hospital_fold_assignment_06B = (
    best_assignment
    .sort_values("group_hospital")
    .reset_index(drop=True)
)

hospital_fold_assignment_06B[
    "assignment_seed"
] = ASSIGNMENT_SEED

fold_summary_06B = (
    hospital_fold_assignment_06B
    .groupby(
        "outer_fold",
        as_index=False
    )
    .agg(
        hospitals=(
            "group_hospital",
            "nunique"
        ),
        patients=(
            "patients",
            "sum"
        ),
        events=(
            "events",
            "sum"
        ),
        nonevents=(
            "nonevents",
            "sum"
        ),
        zero_event_hospitals=(
            "events",
            lambda values: int(
                (values == 0).sum()
            )
        ),
    )
)

fold_summary_06B["event_rate"] = (
    fold_summary_06B["events"]
    / fold_summary_06B["patients"]
)

fold_summary_06B[
    "patient_deviation_from_target"
] = (
    fold_summary_06B["patients"]
    - target_patients
)

fold_summary_06B[
    "event_deviation_from_target"
] = (
    fold_summary_06B["events"]
    - target_events
)

fold_summary_06B[
    "patient_relative_deviation"
] = (
    fold_summary_06B[
        "patient_deviation_from_target"
    ]
    / target_patients
)

fold_summary_06B[
    "event_relative_deviation"
] = (
    fold_summary_06B[
        "event_deviation_from_target"
    ]
    / target_events
)

# ------------------------------------------------------------
# 6. Kesin bütünlük kontrolleri
# ------------------------------------------------------------

if len(hospital_fold_assignment_06B) != 198:
    raise RuntimeError(
        "Fold atamasında 198 hastane bulunmuyor."
    )

if (
    hospital_fold_assignment_06B[
        "group_hospital"
    ].duplicated().any()
):
    raise RuntimeError(
        "Bir hastane birden fazla dış kata atanmış."
    )

if (
    hospital_fold_assignment_06B[
        "outer_fold"
    ].isna().any()
):
    raise RuntimeError(
        "Dış kat ataması eksik hastane var."
    )

if set(
    hospital_fold_assignment_06B[
        "outer_fold"
    ].unique()
) != {1, 2, 3, 4, 5}:
    raise RuntimeError(
        "Dış kat numaraları 1–5 değil."
    )

if int(
    fold_summary_06B["patients"].sum()
) != 58491:
    raise RuntimeError(
        "Fold toplam hasta sayısı hatalı."
    )

if int(
    fold_summary_06B["events"].sum()
) != 3032:
    raise RuntimeError(
        "Fold toplam olay sayısı hatalı."
    )

if int(
    fold_summary_06B["hospitals"].sum()
) != 198:
    raise RuntimeError(
        "Fold toplam hastane sayısı hatalı."
    )

if not fold_summary_06B[
    "hospitals"
].between(39, 40).all():
    raise RuntimeError(
        "Her fold 39 veya 40 hastane içermiyor."
    )

if (
    fold_summary_06B[
        "events"
    ] <= 0
).any():
    raise RuntimeError(
        "Olay içermeyen dış validasyon katı var."
    )

# ------------------------------------------------------------
# 7. CSV ve SHA-256 kilidi
# ------------------------------------------------------------

assignment_path = os.path.join(
    OUTPUT_DIR,
    "06B_locked_hospital_outer_folds_v1.csv"
)

hospital_fold_assignment_06B.to_csv(
    assignment_path,
    index=False
)

with open(
    assignment_path,
    "rb"
) as file_handle:
    fold_sha256 = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

sha_path = os.path.join(
    OUTPUT_DIR,
    "06B_locked_hospital_outer_folds_v1_SHA256.txt"
)

with open(
    sha_path,
    "w",
    encoding="utf-8"
) as file_handle:
    file_handle.write(
        fold_sha256 + "\n"
    )

fold_summary_path = os.path.join(
    OUTPUT_DIR,
    "06B_outer_fold_summary.csv"
)

fold_summary_06B.to_csv(
    fold_summary_path,
    index=False
)

# ------------------------------------------------------------
# 8. BigQuery aggregate fold tablosunu oluştur
# ------------------------------------------------------------

bq_assignment = (
    hospital_fold_assignment_06B[
        [
            "group_hospital",
            "outer_fold",
            "patients",
            "events",
            "nonevents",
            "event_rate",
            "assignment_seed",
        ]
    ].copy()
)

table_id = (
    f"{TARGET_DATASET}."
    "hospital_outer_fold_v1"
)

job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "group_hospital",
            "STRING",
            mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "outer_fold",
            "INTEGER",
            mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "patients",
            "INTEGER",
            mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "events",
            "INTEGER",
            mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "nonevents",
            "INTEGER",
            mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "event_rate",
            "FLOAT",
            mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "assignment_seed",
            "INTEGER",
            mode="REQUIRED"
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    )
)

print(
    "Uploading aggregate hospital fold table:",
    table_id
)

load_job = client.load_table_from_dataframe(
    bq_assignment,
    table_id,
    job_config=job_config,
    location=BQ_LOCATION
)

load_job.result()

# ------------------------------------------------------------
# 9. Core ve extended fold görünümlerini oluştur
# ------------------------------------------------------------

fold_view_sql = f"""
CREATE OR REPLACE VIEW
  `{TARGET_DATASET}.feature_matrix_core_outerfold_v1` AS

SELECT
  m.*,
  f.outer_fold

FROM `{TARGET_DATASET}.feature_matrix_core_view_v1` AS m

INNER JOIN `{TARGET_DATASET}.hospital_outer_fold_v1` AS f
  USING (group_hospital);


CREATE OR REPLACE VIEW
  `{TARGET_DATASET}.feature_matrix_extended_outerfold_v1` AS

SELECT
  m.*,
  f.outer_fold

FROM `{TARGET_DATASET}.feature_matrix_extended_view_v1` AS m

INNER JOIN `{TARGET_DATASET}.hospital_outer_fold_v1` AS f
  USING (group_hospital);
"""

client.query(
    fold_view_sql,
    location=BQ_LOCATION
).result()

# ------------------------------------------------------------
# 10. Göster
# ------------------------------------------------------------

print("\n06B LOCKED OUTER FOLD SUMMARY")
display(fold_summary_06B)

print("\nBalance score:")
print(best_score)

print("\nFold assignment SHA-256:")
print(fold_sha256)

print("\nSaved:")
print(assignment_path)
print(fold_summary_path)
print(sha_path)

print(
    "\n06B PASS: Five disjoint hospital-level "
    "outer folds were locked."
)

In [ ]:
import os
import pandas as pd
from IPython.display import display

# ============================================================
# 06C — Locked outer-fold row and schema integrity audit
# ============================================================

TARGET_DATASET = globals().get(
    "TARGET_DATASET",
    f"{PROJECT_ID}.aki_jcmc_v2"
)

BQ_LOCATION = globals().get(
    "BQ_LOCATION",
    "US"
)

OUTPUT_DIR = globals().get(
    "OUTPUT_DIR",
    f"{OUTPUT_ROOT}/"
    "04_FEATURE_MATRIX_OUTPUTS"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Core ve extended dış-kat görünümlerinin bütünlüğü
# ------------------------------------------------------------

SQL_06C_INTEGRITY = f"""
WITH core AS (
  SELECT
    id_row,
    group_hospital,
    label_stage23,
    outer_fold
  FROM `{TARGET_DATASET}.feature_matrix_core_outerfold_v1`
),

extended AS (
  SELECT
    id_row,
    group_hospital,
    label_stage23,
    outer_fold
  FROM `{TARGET_DATASET}.feature_matrix_extended_outerfold_v1`
),

core_hospital_fold_check AS (
  SELECT
    group_hospital,
    COUNT(DISTINCT outer_fold) AS fold_count
  FROM core
  GROUP BY group_hospital
),

extended_hospital_fold_check AS (
  SELECT
    group_hospital,
    COUNT(DISTINCT outer_fold) AS fold_count
  FROM extended
  GROUP BY group_hospital
),

row_mismatch AS (
  SELECT
    COUNT(*) AS mismatched_rows
  FROM core AS c
  FULL OUTER JOIN extended AS e
    ON e.id_row = c.id_row
  WHERE
    c.id_row IS NULL
    OR e.id_row IS NULL
    OR c.group_hospital IS DISTINCT FROM e.group_hospital
    OR c.label_stage23 IS DISTINCT FROM e.label_stage23
    OR c.outer_fold IS DISTINCT FROM e.outer_fold
)

SELECT
  (SELECT COUNT(*) FROM core)
    AS core_rows,

  (SELECT COUNT(DISTINCT id_row) FROM core)
    AS core_distinct_rows,

  (SELECT COUNT(DISTINCT group_hospital) FROM core)
    AS core_hospitals,

  (SELECT COUNT(DISTINCT outer_fold) FROM core)
    AS core_outer_folds,

  (SELECT COUNTIF(label_stage23 = 1) FROM core)
    AS core_events,

  (SELECT COUNTIF(label_stage23 = 0) FROM core)
    AS core_nonevents,

  (SELECT COUNTIF(label_stage23 IS NULL) FROM core)
    AS core_missing_labels,

  (SELECT COUNTIF(outer_fold IS NULL) FROM core)
    AS core_missing_outer_fold,

  (
    SELECT COUNT(*)
    FROM core_hospital_fold_check
    WHERE fold_count != 1
  ) AS core_hospitals_in_multiple_folds,

  (SELECT COUNT(*) FROM extended)
    AS extended_rows,

  (SELECT COUNT(DISTINCT id_row) FROM extended)
    AS extended_distinct_rows,

  (SELECT COUNT(DISTINCT group_hospital) FROM extended)
    AS extended_hospitals,

  (SELECT COUNT(DISTINCT outer_fold) FROM extended)
    AS extended_outer_folds,

  (SELECT COUNTIF(label_stage23 = 1) FROM extended)
    AS extended_events,

  (SELECT COUNTIF(label_stage23 = 0) FROM extended)
    AS extended_nonevents,

  (SELECT COUNTIF(label_stage23 IS NULL) FROM extended)
    AS extended_missing_labels,

  (SELECT COUNTIF(outer_fold IS NULL) FROM extended)
    AS extended_missing_outer_fold,

  (
    SELECT COUNT(*)
    FROM extended_hospital_fold_check
    WHERE fold_count != 1
  ) AS extended_hospitals_in_multiple_folds,

  (SELECT mismatched_rows FROM row_mismatch)
    AS core_extended_row_assignment_mismatches;
"""

print("Running: 06C_outer_fold_integrity_audit.sql")

integrity_06C = client.query(
    SQL_06C_INTEGRITY,
    location=BQ_LOCATION
).to_dataframe()

display(integrity_06C)

# ------------------------------------------------------------
# 2. Fold başına test ve eğitim büyüklükleri
# ------------------------------------------------------------

SQL_06C_FOLD_SUMMARY = f"""
WITH core AS (
  SELECT
    id_row,
    group_hospital,
    label_stage23,
    outer_fold
  FROM `{TARGET_DATASET}.feature_matrix_core_outerfold_v1`
),

hospital_events AS (
  SELECT
    outer_fold,
    group_hospital,
    COUNT(*) AS patients,
    COUNTIF(label_stage23 = 1) AS events
  FROM core
  GROUP BY outer_fold, group_hospital
),

zero_event_summary AS (
  SELECT
    outer_fold,
    COUNTIF(events = 0) AS zero_event_hospitals
  FROM hospital_events
  GROUP BY outer_fold
),

fold_summary AS (
  SELECT
    outer_fold,
    COUNT(*) AS test_patients,
    COUNT(DISTINCT group_hospital) AS test_hospitals,
    COUNTIF(label_stage23 = 1) AS test_events,
    COUNTIF(label_stage23 = 0) AS test_nonevents,
    SAFE_DIVIDE(
      COUNTIF(label_stage23 = 1),
      COUNT(*)
    ) AS test_event_rate
  FROM core
  GROUP BY outer_fold
)

SELECT
  f.outer_fold,

  f.test_hospitals,
  f.test_patients,
  f.test_events,
  f.test_nonevents,
  f.test_event_rate,

  z.zero_event_hospitals,

  198 - f.test_hospitals
    AS training_hospitals,

  58491 - f.test_patients
    AS training_patients,

  3032 - f.test_events
    AS training_events,

  55459 - f.test_nonevents
    AS training_nonevents,

  SAFE_DIVIDE(
    3032 - f.test_events,
    58491 - f.test_patients
  ) AS training_event_rate

FROM fold_summary AS f

INNER JOIN zero_event_summary AS z
  USING (outer_fold)

ORDER BY f.outer_fold;
"""

fold_audit_06C = client.query(
    SQL_06C_FOLD_SUMMARY,
    location=BQ_LOCATION
).to_dataframe()

print("\n06C OUTER FOLD TRAIN–TEST SUMMARY")
display(fold_audit_06C)

# ------------------------------------------------------------
# 3. Kesin geçiş kontrolleri
# ------------------------------------------------------------

expected_integrity = {
    "core_rows": 58491,
    "core_distinct_rows": 58491,
    "core_hospitals": 198,
    "core_outer_folds": 5,
    "core_events": 3032,
    "core_nonevents": 55459,
    "core_missing_labels": 0,
    "core_missing_outer_fold": 0,
    "core_hospitals_in_multiple_folds": 0,

    "extended_rows": 58491,
    "extended_distinct_rows": 58491,
    "extended_hospitals": 198,
    "extended_outer_folds": 5,
    "extended_events": 3032,
    "extended_nonevents": 55459,
    "extended_missing_labels": 0,
    "extended_missing_outer_fold": 0,
    "extended_hospitals_in_multiple_folds": 0,

    "core_extended_row_assignment_mismatches": 0,
}

audit_row = integrity_06C.iloc[0]

for field, expected_value in expected_integrity.items():
    actual_value = int(audit_row[field])

    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, "
            f"beklenen={expected_value}"
        )

if len(fold_audit_06C) != 5:
    raise RuntimeError(
        f"Beş dış kat bekleniyordu; "
        f"{len(fold_audit_06C)} bulundu."
    )

if set(
    fold_audit_06C["outer_fold"].astype(int)
) != {1, 2, 3, 4, 5}:
    raise RuntimeError(
        "Dış kat numaraları 1–5 değil."
    )

if not fold_audit_06C[
    "test_hospitals"
].between(39, 40).all():
    raise RuntimeError(
        "Test katlarında 39–40 hastane koşulu sağlanmadı."
    )

if not (
    fold_audit_06C["zero_event_hospitals"] == 6
).all():
    raise RuntimeError(
        "Her test katında altı sıfır olaylı hastane yok."
    )

if (
    fold_audit_06C["test_events"] <= 0
).any():
    raise RuntimeError(
        "Olay içermeyen dış test katı var."
    )

if (
    fold_audit_06C["training_events"] <= 0
).any():
    raise RuntimeError(
        "Olay içermeyen dış eğitim seti var."
    )

# ------------------------------------------------------------
# 4. Toplulaştırılmış sonuçları kaydet
# ------------------------------------------------------------

integrity_path = os.path.join(
    OUTPUT_DIR,
    "06C_outer_fold_integrity_audit.csv"
)

fold_summary_path = os.path.join(
    OUTPUT_DIR,
    "06C_outer_fold_train_test_summary.csv"
)

integrity_06C.to_csv(
    integrity_path,
    index=False
)

fold_audit_06C.to_csv(
    fold_summary_path,
    index=False
)

print("\n06C PASS: No hospital or patient crosses outer folds.")
print("Core and extended fold assignments are identical.")
print("Saved aggregate files to:", OUTPUT_DIR)

In [ ]:
SQL_06 = '-- Shareable aggregate integrity audit. No patient-level rows are returned.\nSELECT\n  COUNT(*) AS matrix_rows,\n  COUNT(DISTINCT patientUnitStayID) AS distinct_unit_stays,\n  COUNT(DISTINCT id_row) AS distinct_row_keys,\n  COUNT(DISTINCT hospitalID) AS hospitals,\n  COUNTIF(label_stage23 = 1) AS events,\n  COUNTIF(label_stage23 = 0) AS nonevents,\n  SAFE_DIVIDE(COUNTIF(label_stage23 = 1), COUNT(*)) AS event_rate,\n  COUNT(*) - COUNT(DISTINCT patientUnitStayID) AS duplicate_stay_rows,\n  COUNTIF(label_stage23 IS NULL) AS missing_labels,\n  COUNTIF(x_reference_creatinine IS NULL) AS missing_reference_creatinine\nFROM `{{TARGET_DATASET}}.feature_matrix_v1`;\n'
result_06 = run_aggregate('06_feature_matrix_integrity_audit.sql', SQL_06, '06_feature_matrix_integrity_audit.csv')

## 07 Feature namespace audit

In [ ]:
SQL_07 = "-- Shareable namespace audit: only x_ columns are predictors.\nSELECT\n  CASE\n    WHEN STARTS_WITH(column_name,'x_') THEN 'predictor'\n    WHEN STARTS_WITH(column_name,'audit_') THEN 'audit_only'\n    WHEN STARTS_WITH(column_name,'id_') THEN 'identifier'\n    WHEN STARTS_WITH(column_name,'group_') THEN 'grouping'\n    WHEN STARTS_WITH(column_name,'label_') THEN 'outcome'\n    ELSE 'internal_identifier'\n  END AS column_role,\n  COUNT(*) AS n_columns\nFROM `{{TARGET_DATASET}}.INFORMATION_SCHEMA.COLUMNS`\nWHERE table_name = 'feature_matrix_v1'\nGROUP BY column_role\nORDER BY column_role;\n"
result_07 = run_aggregate('07_feature_namespace_audit.sql', SQL_07, '07_feature_namespace_audit.csv')

## 08 Predictor coverage audit

In [ ]:
schema_sql = f'''SELECT column_name, data_type
FROM `{TARGET_DATASET}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'feature_matrix_v1'
  AND STARTS_WITH(column_name, 'x_')
ORDER BY ordinal_position'''
cols = client.query(schema_sql, location=BQ_LOCATION).to_dataframe()
parts = []
for row in cols.itertuples(index=False):
    c = row.column_name
    parts.append(f'''SELECT '{c}' AS predictor,
      COUNTIF(`{c}` IS NOT NULL) AS nonmissing,
      COUNT(*) AS total,
      SAFE_DIVIDE(COUNTIF(`{c}` IS NOT NULL), COUNT(*)) AS coverage
    FROM `{TARGET_DATASET}.feature_matrix_v1`''')
coverage_sql = '\nUNION ALL\n'.join(parts) + '\nORDER BY coverage DESC, predictor'
result_08 = run_aggregate('08_predictor_coverage_audit.sql', coverage_sql, '08_predictor_coverage_audit.csv')
print('Predictor columns:', len(cols))


## 09 Restricted-name leakage audit

In [ ]:
restricted_sql = f'''SELECT column_name
FROM `{TARGET_DATASET}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'feature_matrix_v1'
  AND STARTS_WITH(column_name, 'x_')
  AND REGEXP_CONTAINS(LOWER(column_name), r'outcome|future|incident|discharge|death|mortality|rrt|dialysis|stage23|observation_class')
ORDER BY column_name'''
result_09 = run_aggregate('09_restricted_predictor_name_audit.sql', restricted_sql, '09_restricted_predictor_name_audit.csv')
if len(result_09) != 0:
    raise RuntimeError('Restricted predictor names detected; stop before modelling.')
print('PASS: no restricted predictor names.')


## 10 Aggregate share package

In [ ]:
share_files = [
    '06_feature_matrix_integrity_audit.csv',
    '07_feature_namespace_audit.csv',
    '08_predictor_coverage_audit.csv',
    '09_restricted_predictor_name_audit.csv',
]
manifest=[]
for name in share_files:
    p=os.path.join(DRIVE_OUTPUT_DIR,name)
    with open(p,'rb') as f: data=f.read()
    manifest.append({'filename':name,'size_bytes':len(data),'sha256':hashlib.sha256(data).hexdigest()})
manifest_path=os.path.join(DRIVE_OUTPUT_DIR,'MANIFEST_FEATURE_MATRIX.json')
with open(manifest_path,'w') as f: json.dump(manifest,f,indent=2)
zip_path=os.path.join(DRIVE_OUTPUT_DIR,'AKI_V2_FEATURE_MATRIX_AGGREGATES.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for name in share_files: z.write(os.path.join(DRIVE_OUTPUT_DIR,name),arcname=name)
    z.write(manifest_path,arcname='MANIFEST_FEATURE_MATRIX.json')
print('Shareable aggregate package:',zip_path)
print('Do not export or upload feature_matrix_v1 or feature_matrix_model_view_v1.')
